In [14]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import pandas as pd
from urllib.parse import quote
from pathlib import Path

RAW_DIR = Path('../Data/Raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK. Raw folder:", RAW_DIR.resolve())


Setup OK. Raw folder: D:\Library Analytics\Data\Raw


In [16]:
base = "https://data.seattle.gov/resource/tmmm-ytt6.csv"

where_clause = (
    "checkoutyear >= 2020 AND checkoutyear <= 2023 "
    "AND materialtype in ('BOOK', 'EBOOK', 'AUDIOBOOK') "
    "AND checkouts >= 5"
)

url = f"{base}?$where={quote(where_clause)}&$limit=2000000"

print("Pulling from API... 1-3 min is normal.")
df = pd.read_csv(url)
print(f"Done. Pulled {len(df):,} rows.")


Pulling from API... 1-3 min is normal.


C:\Users\vardh\AppData\Local\Temp\ipykernel_32296\4133834653.py:12: DtypeWarning: Columns (7: isbn) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


Done. Pulled 1,342,223 rows.


In [17]:
raw_path = RAW_DIR / 'seattle_checkouts_2020_2023.csv'
df.to_csv(raw_path, index=False)
print(f"Saved to: {raw_path.resolve()}")
print(f"File size: {raw_path.stat().st_size / 1e6:.1f} MB")

Saved to: D:\Library Analytics\Data\Raw\seattle_checkouts_2020_2023.csv
File size: 310.4 MB


In [18]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
print("\nMemory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print("\nYear range:", df['checkoutyear'].min(), "-", df['checkoutyear'].max())
print("\nMaterialType counts:")
print(df['materialtype'].value_counts())
print("\nHead:")
df.head()


Shape: (1342223, 12)

Columns: ['usageclass', 'checkouttype', 'materialtype', 'checkoutyear', 'checkoutmonth', 'checkouts', 'title', 'isbn', 'creator', 'subjects', 'publisher', 'publicationyear']

Dtypes:
usageclass            str
checkouttype          str
materialtype          str
checkoutyear        int64
checkoutmonth       int64
checkouts           int64
title                 str
isbn               object
creator               str
subjects              str
publisher             str
publicationyear       str
dtype: object

Memory (MB): 891.9

Year range: 2020 - 2023

MaterialType counts:
materialtype
EBOOK        515600
BOOK         446506
AUDIOBOOK    380117
Name: count, dtype: int64

Head:


,usageclass,checkouttype,materialtype,checkoutyear,checkoutmonth,checkouts,title,isbn,creator,subjects,publisher,publicationyear
0,Digital,OverDrive,EBOOK,2020,8,5,I Can Cook Vegan,NaN,Isa Chandra Moskowitz,"Cooking & Food, Nonfiction",ABRAMS,2019
1,Digital,OverDrive,EBOOK,2020,8,5,The Outside,NaN,Ada Hoffmann,"Fiction, Science Fiction","Random House, Inc.",2019
2,Digital,OverDrive,EBOOK,2020,8,16,Blindness,NaN,José Saramago,"Fiction, Literature",Houghton Mifflin Harcourt Trade and Reference,2011
3,Digital,OverDrive,EBOOK,2020,8,5,That Good Night,NaN,Richard Probert,"Fantasy, Fiction",Independent Publishers Group,2016
4,Digital,OverDrive,EBOOK,2020,8,5,Hunting Game,NaN,Helene Tursten,"Fiction, Mystery","Random House, Inc.",2019


In [19]:
# 1. Missingness
print("Nulls per column:")
print(df.isnull().sum())
print("\n% null per column:")
print((df.isnull().sum() / len(df) * 100).round(2))

# 2. Duplicates on the assumed grain
grain_cols = ['title', 'materialtype', 'checkouttype', 'checkoutyear', 'checkoutmonth']
dup_count = df.duplicated(subset=grain_cols).sum()
print(f"\nDuplicates on grain {grain_cols}: {dup_count:,}")

# 3. Checkouts value sanity
print("\nCheckouts summary:")
print(df['checkouts'].describe())

# 4. Cardinality of categoricals
print("\nUnique values in low-card categoricals:")
for col in ['usageclass', 'checkouttype', 'materialtype']:
    print(f"  {col}: {df[col].nunique()} — {df[col].unique().tolist()}")

Nulls per column:
usageclass              0
checkouttype            0
materialtype            0
checkoutyear            0
checkoutmonth           0
checkouts               0
title                   0
isbn               699902
creator             15261
subjects             1788
publisher             977
publicationyear       937
dtype: int64

% null per column:
usageclass          0.00
checkouttype        0.00
materialtype        0.00
checkoutyear        0.00
checkoutmonth       0.00
checkouts           0.00
title               0.00
isbn               52.14
creator             1.14
subjects            0.13
publisher           0.07
publicationyear     0.07
dtype: float64

Duplicates on grain ['title', 'materialtype', 'checkouttype', 'checkoutyear', 'checkoutmonth']: 8,918

Checkouts summary:
count    1.342223e+06
mean     1.225693e+01
std      2.288047e+01
min      5.000000e+00
25%      6.000000e+00
50%      7.000000e+00
75%      1.100000e+01
max      4.903000e+03
Name: checkouts, dtype:

In [20]:
# Step 3: Monthly checkout volume over time
#
# Each row is one (title x edition x format x month) record, and `checkouts`
# is that month's SUM for the record. Volume is therefore df['checkouts'].sum(),
# never len(df). Rows are not deduplicated: a title appearing twice in a month
# is two editions, and both contribute real checkouts.

import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})

# First-of-month timestamp, e.g. 2020-08 -> 2020-08-01
df['date'] = pd.to_datetime(
    dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
)

# Total checkouts per month
monthly_total = (
    df.groupby('date', as_index=False)['checkouts'].sum()
      .rename(columns={'checkouts': 'total_checkouts'})
)

# Monthly checkouts split into Digital vs Physical
monthly_by_usage = df.pivot_table(
    index='date', columns='usageclass', values='checkouts',
    aggfunc='sum', fill_value=0,
)
monthly_by_usage.columns.name = None

# Yearly checkouts split into Digital vs Physical
yearly_by_usage = df.pivot_table(
    index='checkoutyear', columns='usageclass', values='checkouts',
    aggfunc='sum', fill_value=0,
)
yearly_by_usage.columns.name = None


def head_tail(frame, n=6):
    """First and last n rows of a frame, stacked for a compact preview."""
    return pd.concat([frame.head(n), frame.tail(n)])


print("Monthly total checkouts (first 6 + last 6 months):")
print(head_tail(monthly_total).to_string(index=False))

print("\nMonthly checkouts, Digital vs Physical (first 6 + last 6 months):")
print(head_tail(monthly_by_usage))

print("\nYearly checkouts by usage class:")
print(yearly_by_usage)

# Sanity checks: every aggregate must add back up to the raw total
total_checkouts = df['checkouts'].sum()
assert monthly_total['total_checkouts'].sum() == total_checkouts
assert monthly_by_usage.to_numpy().sum() == total_checkouts
assert yearly_by_usage.to_numpy().sum() == total_checkouts

print("\nSanity checks:")
print(f"  Total checkouts: {total_checkouts:,}")
print(f"  Date range:      {df['date'].min():%Y-%m} to {df['date'].max():%Y-%m}")
print(f"  Unique months:   {df['date'].nunique()} (expected 48)")

Monthly total checkouts (first 6 + last 6 months):
      date  total_checkouts
2020-01-01           407826
2020-02-01           363290
2020-03-01           282791
2020-04-01           211941
2020-05-01           239531
2020-06-01           238837
2023-07-01           435738
2023-08-01           443353
2023-09-01           411776
2023-10-01           429901
2023-11-01           411387
2023-12-01           424246

Monthly checkouts, Digital vs Physical (first 6 + last 6 months):
            Digital  Physical
date                         
2020-01-01   204991    202835
2020-02-01   182380    180910
2020-03-01   187933     94858
2020-04-01   211941         0
2020-05-01   239531         0
2020-06-01   238837         0
2023-07-01   306650    129088
2023-08-01   310397    132956
2023-09-01   293048    118728
2023-10-01   307842    122059
2023-11-01   297538    113849
2023-12-01   319733    104513

Yearly checkouts by usage class:
              Digital  Physical
checkoutyear                   


In [21]:
# Step 3b: Investigate the COVID physical checkout gap
#
# Step 3 showed Physical checkouts falling to 0 in April 2020. This cell traces
# the full closure and recovery arc. Values are monthly sums of `checkouts`,
# not row counts.
#
# Caveat: the download kept only records with checkouts >= 5, so a 0 here means
# no physical record reached that floor in the month, not literally zero loans.

import pandas as pd
from pathlib import Path

# Rebuild the monthly pivot only if Step 3 hasn't run in this session
if 'monthly_by_usage' not in globals():
    RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
    df = pd.read_csv(RAW_PATH, dtype={'isbn': str})
    df['date'] = pd.to_datetime(
        dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
    )
    monthly_by_usage = df.pivot_table(
        index='date', columns='usageclass', values='checkouts',
        aggfunc='sum', fill_value=0,
    )
    monthly_by_usage.columns.name = None

physical = monthly_by_usage['Physical']

# 1. Every month of 2020
print("Physical checkouts by month, 2020:")
print(physical.loc['2020'].to_frame().to_string())

# 2. When did physical checkouts come back?
zero_months = physical[physical == 0]
if zero_months.empty:
    print("\nPhysical checkouts never hit 0 in 2020-2023.")
else:
    closure_start = zero_months.index.min()
    after_closure = physical.loc[closure_start:]
    reopened = after_closure[after_closure > 0]
    if reopened.empty:
        print(f"\nPhysical checkouts hit 0 in {closure_start:%Y-%m} and never returned.")
    else:
        first_return = reopened.index[0]
        months_at_zero = (after_closure.index < first_return).sum()
        print(f"\nFirst zero month:   {closure_start:%Y-%m}")
        print(f"First return month: {first_return:%Y-%m} "
              f"({reopened.iloc[0]:,} physical checkouts)")
        print(f"Months at zero:     {months_at_zero}")

# 3. Full closure + recovery arc, 2020-2021
arc = physical.loc['2020':'2021']
print(f"\nPhysical checkouts by month, 2020-2021 ({len(arc)} months):")
print(arc.to_frame().to_string())

Physical checkouts by month, 2020:
            Physical
date                
2020-01-01    202835
2020-02-01    180910
2020-03-01     94858
2020-04-01         0
2020-05-01         0
2020-06-01         0
2020-07-01         0
2020-08-01       641
2020-09-01     18310
2020-10-01     32040
2020-11-01     35237
2020-12-01     44840

First zero month:   2020-04
First return month: 2020-08 (641 physical checkouts)
Months at zero:     4

Physical checkouts by month, 2020-2021 (24 months):
            Physical
date                
2020-01-01    202835
2020-02-01    180910
2020-03-01     94858
2020-04-01         0
2020-05-01         0
2020-06-01         0
2020-07-01         0
2020-08-01       641
2020-09-01     18310
2020-10-01     32040
2020-11-01     35237
2020-12-01     44840
2021-01-01     54453
2021-02-01     53675
2021-03-01     69885
2021-04-01     64222
2021-05-01     66551
2021-06-01     91595
2021-07-01    117588
2021-08-01    119109
2021-09-01    128154
2021-10-01    131173
2021-11-01

In [22]:
# Step 4: Top titles, authors, and format-share evolution
#
# `checkouts` is a per-record monthly sum, so every ranking here sums that
# column rather than counting rows. Editions are collapsed with groupby('title')
# / groupby('creator'); the raw rows are never deduplicated.
#
# Caveats, both visible in the output below:
#   - Title and creator strings follow different conventions per source. Horizon
#     (physical) writes "Lessons in chemistry / Bonnie Garmus." and "Willems, Mo",
#     OverDrive (digital) writes "Lessons in Chemistry: A Novel" and "Mo Willems",
#     and audiobooks append "(unabridged)". So these groupbys collapse editions
#     within a source, not the same work across formats. Normalising that is a
#     cleaning step of its own.
#   - Titles are matched on text alone, so two unrelated works sharing a title
#     would merge. `creator` is 1.14% null, dropped only for the author table.

import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})


def ranked(frame):
    """Number a frame 1..n and label the index 'rank' for display."""
    frame = frame.reset_index(drop=True)
    frame.index = range(1, len(frame) + 1)
    frame.index.name = 'rank'
    return frame


# 1. Top titles overall, all editions and formats combined
title_totals = df.groupby('title')['checkouts'].sum().sort_values(ascending=False)
top_titles = title_totals.head(15)

# The format each of those titles was borrowed in most often
primary_materialtype = (
    df[df['title'].isin(top_titles.index)]
      .groupby(['title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('title')          # highest-checkout format per title
      .set_index('title')['materialtype']
)
top_titles = ranked(
    top_titles.rename('total_checkouts').reset_index()
              .assign(primary_materialtype=lambda d: d['title'].map(primary_materialtype))
)

# 2. Top titles within each format
title_by_format = (
    df.groupby(['materialtype', 'title'])['checkouts'].sum()
      .rename('total_checkouts').reset_index()
      .sort_values(['materialtype', 'total_checkouts'], ascending=[True, False])
)
top_by_format = (
    title_by_format.groupby('materialtype').head(5)
                   .assign(rank=lambda d: d.groupby('materialtype').cumcount() + 1)
                   [['materialtype', 'rank', 'title', 'total_checkouts']]
                   .reset_index(drop=True)
)

# 3. Top authors (creator is null on 1.14% of rows, so drop those first)
top_authors = ranked(
    df.dropna(subset=['creator'])
      .groupby('creator')
      .agg(total_checkouts=('checkouts', 'sum'), unique_titles=('title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(15)
      .reset_index()
)

# 4. Format share of checkouts per year (% of that year's total)
year_by_format = df.pivot_table(
    index='checkoutyear', columns='materialtype', values='checkouts',
    aggfunc='sum', fill_value=0,
).reindex(columns=['BOOK', 'EBOOK', 'AUDIOBOOK'])
year_by_format.columns.name = None
format_share = (
    year_by_format.div(year_by_format.sum(axis=1), axis=0) * 100
).round(1)

with pd.option_context('display.max_colwidth', 55, 'display.width', 140):
    print("=" * 78)
    print("1. TOP 15 TITLES, 2020-2023 (all formats and editions combined)")
    print("=" * 78)
    print(top_titles.to_string())

    print("\n" + "=" * 78)
    print("2. TOP 5 TITLES PER FORMAT")
    print("=" * 78)
    for materialtype, group in top_by_format.groupby('materialtype'):
        print(f"\n{materialtype}:")
        print(group[['rank', 'title', 'total_checkouts']].to_string(index=False))

    print("\n" + "=" * 78)
    print("3. TOP 15 AUTHORS, 2020-2023")
    print("=" * 78)
    print(top_authors.to_string())

    print("\n" + "=" * 78)
    print("4. FORMAT SHARE OF CHECKOUTS BY YEAR (% of year total)")
    print("=" * 78)
    print(format_share.to_string())

    # Shares are rounded to 1dp, so a row can land on 100.1 instead of 100.0
    print("\nRow totals (sanity check, ~100 each; rounding drift is expected):")
    print(format_share.sum(axis=1).to_string())

1. TOP 15 TITLES, 2020-2023 (all formats and editions combined)
                                                                                                      title  total_checkouts primary_materialtype
rank                                                                                                                                             
1     Braiding Sweetgrass: Indigenous Wisdom, Scientific Knowledge and the Teachings of Plants (unabridged)            17903            AUDIOBOOK
2                                                101 Essays That Will Change the Way You Think (unabridged)            12830            AUDIOBOOK
3                      Maybe You Should Talk to Someone: A Therapist, HER Therapist, and Our Lives Revealed            12163                EBOOK
4                                                               So You Want to Talk about Race (Unabridged)            11783            AUDIOBOOK
5                                                           

In [37]:
# Step 4b: Title/creator normalization + updated top-N
#
# Step 4 showed the same work splitting across sources, because Horizon
# (physical) and OverDrive (digital) catalogue differently:
#   physical  "Lessons in chemistry / Bonnie Garmus."   creator "Willems, Mo"
#   ebook     "Lessons in Chemistry: A Novel"           creator "Mo Willems"
#   audiobook "Lessons in Chemistry: A Novel (Unabridged)"
#
# Revision (second pass). The first version missed a systematic print/digital
# split: Horizon writes subtitles with a space before the colon
# ("Sapiens : a brief history of humankind"), OverDrive without one ("Sapiens: A
# Brief History of Humankind"), so print and digital editions of the same work
# landed on different keys. It also only stripped ": a novel" / ": a memoir"
# when 3+ words remained (so "educated: a memoir" never met "educated"), and it
# left series suffixes ("dune: dune series, book 1") in place. The fix collapses
# " : " and whitespace runs first, drops the 3-word guard, widens the stock
# subtitle list and strips series suffixes.
#
# Merge rates (distinct raw values -> distinct cleaned values):
#   titles    first version 178,548 -> 124,915 (30.0% merged)
#             this version  178,548 -> 103,856 (41.8% merged)
#   creators  first version  55,246 ->  41,325 (25.2% merged)
#             this version   55,246 ->  41,239 (25.4% merged)
#
# clean_title rules, applied in this order:
#   1. collapse " : " to ": "          — Horizon's spaced subtitle colon
#   2. collapse whitespace runs to one space
#   3. lowercase                       — case differs between the two sources
#   4. drop "(unabridged)"/"(abridged)" and any [bracketed] format marker
#      such as "[videorecording]" or "[sound recording]"
#   5. drop a trailing " / author." catalogue suffix (physical records only)
#   6. drop a trailing series suffix: ": <any> series, book N" or ": book N"
#   7. drop a trailing stock subtitle, whatever the base length: ": a novel",
#      ": a memoir", ": a story", ": a true story", ": the unabridged edition"
#   8. trim whitespace, then trailing colons and periods
#
# clean_creator rules:
#   whitespace runs collapse to one space; "Surname, Forename" and "Surname,
#   Forename, 1966-" become "Forename Surname", birth/death years are dropped,
#   and the result is title-cased. Names without a comma are left as written.
#   A result that is empty or on CREATOR_BLOCKLIST (publishers and placeholders,
#   matched case-insensitively) becomes NaN. Rows are kept either way: only the
#   creator value is blanked, so checkout totals are unchanged.
#
# Known limitations, deliberately left in:
#   - Stock-subtitle stripping covers a fixed list. Other subtitle styles
#     (": a thriller", ": stories") still split, and a work genuinely titled
#     "<something>: A Novel" loses its subtitle.
#   - A print record that omits a subtitle the digital record carries still
#     splits: "braiding sweetgrass" (print) vs "braiding sweetgrass: indigenous
#     wisdom, ..." (digital). Only exact stock phrases are stripped.
#   - .title() mangles internal capitals: "McDonald" becomes "Mcdonald".
#   - Only 4-digit year forms like ", 1966-" or ", 1900-1985" are removed.
#   - Two different works sharing a cleaned title still merge.
# The original `title` and `creator` columns are kept alongside the cleaned ones.

import re
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
df = pd.read_csv(RAW_PATH, dtype={'isbn': str})

SPACED_COLON_RE = re.compile(r'\s+:\s+')
WHITESPACE_RE = re.compile(r'\s+')
ABRIDGED_RE = re.compile(r'\s*\((?:un)?abridged\)')
BRACKETED_RE = re.compile(r'\s*\[[^\]]*\]')
CATALOGUE_SUFFIX_RE = re.compile(r'\s+/\s+.*$')
SERIES_SUFFIX_RE = re.compile(r':\s*(?:[^:]*\bseries,\s*)?book\s+\d+$', re.IGNORECASE)
STOCK_SUBTITLE_RE = re.compile(
    r':\s*(?:a novel|a memoir|a story|a true story|the unabridged edition)$', re.IGNORECASE
)
YEAR_RANGE_RE = re.compile(r'^\d{4}\??-(?:\d{4})?\.?$')
CREATOR_BLOCKLIST = {'dk', 'disney', 'various', 'anonymous'}

# Distinct counts from the first version of this cell, for comparison
FIRST_VERSION = {'titles': 124_915, 'creators': 41_325}


def clean_title_text(title):
    """Normalise one title string (see the rules in the header comment)."""
    if not isinstance(title, str):
        return title
    text = SPACED_COLON_RE.sub(': ', title)
    text = WHITESPACE_RE.sub(' ', text)
    text = text.lower()
    text = ABRIDGED_RE.sub('', text)
    text = BRACKETED_RE.sub('', text)
    text = CATALOGUE_SUFFIX_RE.sub('', text)
    text = text.strip().rstrip('.').strip()
    text = SERIES_SUFFIX_RE.sub('', text)
    text = STOCK_SUBTITLE_RE.sub('', text)
    text = text.strip().rstrip(':.').strip()
    return text or title.lower().strip()


def clean_creator_text(creator):
    """Turn "Surname, Forename, 1966-" into "Forename Surname"; blank blocklisted names."""
    if not isinstance(creator, str):
        return creator
    name = WHITESPACE_RE.sub(' ', creator).strip()
    if ',' in name:
        parts = [p.strip() for p in name.split(',')]
        parts = [p for p in parts if p and not YEAR_RANGE_RE.match(p)]
        if len(parts) >= 2:
            name = f"{parts[1]} {parts[0]}"   # reverse the first two parts
        elif parts:
            name = parts[0]
    name = name.strip().title()
    if not name or name.lower() in CREATOR_BLOCKLIST:
        return np.nan
    return name


def apply_to_unique(series, func):
    """Run func once per distinct value, then map back — far cheaper per row."""
    distinct = series.dropna().unique()
    lookup = pd.Series([func(v) for v in distinct], index=distinct, dtype=object)
    return series.map(lookup)


df['clean_title'] = apply_to_unique(df['title'], clean_title_text)
df['clean_creator'] = apply_to_unique(df['creator'], clean_creator_text)


def ranked(frame):
    """Number a frame 1..n and label the index 'rank' for display."""
    frame = frame.reset_index(drop=True)
    frame.index = range(1, len(frame) + 1)
    frame.index.name = 'rank'
    return frame


# Top titles on the cleaned key
clean_title_totals = (
    df.groupby('clean_title')['checkouts'].sum().sort_values(ascending=False)
)
top_clean_titles = clean_title_totals.head(15)
primary_materialtype = (
    df[df['clean_title'].isin(top_clean_titles.index)]
      .groupby(['clean_title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('clean_title')      # highest-checkout format per title
      .set_index('clean_title')['materialtype']
)
top_clean_titles = ranked(
    top_clean_titles.rename('total_checkouts').reset_index()
                    .assign(primary_materialtype=lambda d: d['clean_title'].map(primary_materialtype))
)

# Top creators on the cleaned key
top_clean_creators = ranked(
    df.dropna(subset=['clean_creator'])
      .groupby('clean_creator')
      .agg(total_checkouts=('checkouts', 'sum'),
           unique_titles=('clean_title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(15)
      .reset_index()
)

# Spot check: every edition of "Where the Crawdads Sing"
crawdads = df[df['title'].str.contains('Crawdads', case=False, na=False)]
crawdads_mapping = (
    crawdads[['title', 'clean_title', 'materialtype']]
    .drop_duplicates()
    .sort_values(['clean_title', 'materialtype', 'title'])
)
crawdads_totals = (
    crawdads.groupby(['clean_title', 'materialtype'])['checkouts'].sum()
            .rename('total_checkouts').reset_index()
            .sort_values(['clean_title', 'total_checkouts'], ascending=[True, False])
)

with pd.option_context('display.max_colwidth', 60, 'display.width', 150):
    print("=" * 78)
    print("1. NORMALISATION EFFECT (distinct values before vs after)")
    print("=" * 78)
    counts = pd.DataFrame(
        {
            'before': [df['title'].nunique(), df['creator'].nunique()],
            'after': [df['clean_title'].nunique(), df['clean_creator'].nunique()],
        },
        index=['titles', 'creators'],
    )
    counts['merged'] = counts['before'] - counts['after']
    counts['% merged'] = (counts['merged'] / counts['before'] * 100).round(1)
    counts['first version'] = [FIRST_VERSION['titles'], FIRST_VERSION['creators']]
    counts['first version % merged'] = (
        (counts['before'] - counts['first version']) / counts['before'] * 100
    ).round(1)
    print(counts.to_string())
    print(f"\nRows whose creator is blank after cleaning: {df['clean_creator'].isna().sum():,} "
          f"(raw nulls: {df['creator'].isna().sum():,})")

    print("\n" + "=" * 78)
    print("2. TOP 15 TITLES ON clean_title")
    print("=" * 78)
    print(top_clean_titles.to_string())

    print("\n" + "=" * 78)
    print("3. TOP 15 CREATORS ON clean_creator")
    print("=" * 78)
    print(top_clean_creators.to_string())

    print("\n" + "=" * 78)
    print("4. SPOT CHECK: 'Crawdads' editions")
    print("=" * 78)
    print(f"\nOriginal -> cleaned ({len(crawdads_mapping)} distinct originals):")
    print(crawdads_mapping.to_string(index=False))
    print("\nCheckouts grouped by (clean_title, materialtype):")
    print(crawdads_totals.to_string(index=False))
    print("\nCombined across formats:")
    print(crawdads.groupby('clean_title')['checkouts'].sum().to_string())

1. NORMALISATION EFFECT (distinct values before vs after)
          before   after  merged  % merged  first version  first version % merged
titles    178548  103856   74692      41.8         124915                    30.0
creators   55246   41239   14007      25.4          41325                    25.2

Rows whose creator is blank after cleaning: 15,972 (raw nulls: 15,261)

2. TOP 15 TITLES ON clean_title
                                                                                   clean_title  total_checkouts primary_materialtype
rank                                                                                                                                
1                                                                      where the crawdads sing            27896            AUDIOBOOK
2     braiding sweetgrass: indigenous wisdom, scientific knowledge and the teachings of plants            27214            AUDIOBOOK
3                                                          

In [38]:
# Step 5a: Save cleaned dataset
#
# Rebuilds the derived columns from Step 3 (`date`) and Step 4b (`clean_title`,
# `clean_creator`) on a fresh read of the raw file, then writes one cleaned CSV.
# The transformations are repeated here rather than imported so that every
# Step 5 cell runs standalone, in any order, after a kernel restart.
#
# No rows are dropped and nothing is deduplicated: the row count out must equal
# the row count in. The original `title` and `creator` columns are kept next to
# the cleaned versions so any ranking can be traced back to source spellings.

import re
import numpy as np
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../Data/Raw/seattle_checkouts_2020_2023.csv')
CLEAN_DIR = Path('../Data/Clean')
CLEAN_PATH = CLEAN_DIR / 'seattle_checkouts_2020_2023_clean.csv'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH, dtype={'isbn': str})
input_rows = len(df)

# --- Step 3 logic: first-of-month timestamp -------------------------------
df['date'] = pd.to_datetime(
    dict(year=df['checkoutyear'], month=df['checkoutmonth'], day=1)
)

# --- Step 4b logic: title and creator normalization (keep in sync with 4b) ---
SPACED_COLON_RE = re.compile(r'\s+:\s+')
WHITESPACE_RE = re.compile(r'\s+')
ABRIDGED_RE = re.compile(r'\s*\((?:un)?abridged\)')
BRACKETED_RE = re.compile(r'\s*\[[^\]]*\]')
CATALOGUE_SUFFIX_RE = re.compile(r'\s+/\s+.*$')
SERIES_SUFFIX_RE = re.compile(r':\s*(?:[^:]*\bseries,\s*)?book\s+\d+$', re.IGNORECASE)
STOCK_SUBTITLE_RE = re.compile(
    r':\s*(?:a novel|a memoir|a story|a true story|the unabridged edition)$', re.IGNORECASE
)
YEAR_RANGE_RE = re.compile(r'^\d{4}\??-(?:\d{4})?\.?$')
CREATOR_BLOCKLIST = {'dk', 'disney', 'various', 'anonymous'}


def clean_title_text(title):
    """Normalise one title string (see the rules in the header comment)."""
    if not isinstance(title, str):
        return title
    text = SPACED_COLON_RE.sub(': ', title)
    text = WHITESPACE_RE.sub(' ', text)
    text = text.lower()
    text = ABRIDGED_RE.sub('', text)
    text = BRACKETED_RE.sub('', text)
    text = CATALOGUE_SUFFIX_RE.sub('', text)
    text = text.strip().rstrip('.').strip()
    text = SERIES_SUFFIX_RE.sub('', text)
    text = STOCK_SUBTITLE_RE.sub('', text)
    text = text.strip().rstrip(':.').strip()
    return text or title.lower().strip()


def clean_creator_text(creator):
    """Turn "Surname, Forename, 1966-" into "Forename Surname"; blank blocklisted names."""
    if not isinstance(creator, str):
        return creator
    name = WHITESPACE_RE.sub(' ', creator).strip()
    if ',' in name:
        parts = [p.strip() for p in name.split(',')]
        parts = [p for p in parts if p and not YEAR_RANGE_RE.match(p)]
        if len(parts) >= 2:
            name = f"{parts[1]} {parts[0]}"   # reverse the first two parts
        elif parts:
            name = parts[0]
    name = name.strip().title()
    if not name or name.lower() in CREATOR_BLOCKLIST:
        return np.nan
    return name


def apply_to_unique(series, func):
    """Run func once per distinct value, then map back — far cheaper per row."""
    distinct = series.dropna().unique()
    lookup = pd.Series([func(v) for v in distinct], index=distinct, dtype=object)
    return series.map(lookup)


df['clean_title'] = apply_to_unique(df['title'], clean_title_text)
df['clean_creator'] = apply_to_unique(df['creator'], clean_creator_text)

# --- Write ----------------------------------------------------------------
df.to_csv(CLEAN_PATH, index=False)
output_rows = len(df)

print(f"Input rows:  {input_rows:,}")
print(f"Output rows: {output_rows:,}")
print(f"Row counts match (no rows dropped or deduplicated): {input_rows == output_rows}")

print(f"\nOutput columns ({len(df.columns)}):")
for column in df.columns:
    print(f"  - {column}")

print(f"\nSaved to:   {CLEAN_PATH.resolve()}")
print(f"File exists: {CLEAN_PATH.exists()}")
print(f"File size:   {CLEAN_PATH.stat().st_size / 1e6:.1f} MB")

Input rows:  1,342,223
Output rows: 1,342,223
Row counts match (no rows dropped or deduplicated): True

Output columns (15):
  - usageclass
  - checkouttype
  - materialtype
  - checkoutyear
  - checkoutmonth
  - checkouts
  - title
  - isbn
  - creator
  - subjects
  - publisher
  - publicationyear
  - date
  - clean_title
  - clean_creator

Saved to:   D:\Library Analytics\Data\Clean\seattle_checkouts_2020_2023_clean.csv
File exists: True
File size:   391.3 MB


In [39]:
# Step 5b: Export aggregation CSVs for Power BI
#
# Reads the cleaned file written by Step 5a rather than any in-memory frame, so
# this cell stands on its own. Five aggregations are written to ../Output/, each
# summing the `checkouts` column — never counting rows.
#
# The two monthly files are padded onto a complete date x category grid. The
# April-July 2020 closure produced no qualifying Physical rows at all, and an
# absent row plots as a gap in the line rather than the zero it really was.
#
# The closing sanity block adds each export back up to the known total and
# checks both grids are full, so a silently dropped row fails loudly here.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
OUTPUT_DIR = Path('../Output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TOTAL = 16_451_540   # established in Step 3
EXPECTED_MONTHS = 48          # 2020-01 through 2023-12

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

all_dates = pd.Index(sorted(df['date'].unique()), name='date')


def monthly_grid(category_column):
    """Sum checkouts per (date, category) and pad onto the full grid.

    Every month gets a row for every category; combinations absent from the
    data — a format nobody borrowed that month — come back as 0, not missing.
    """
    aggregated = df.groupby(['date', category_column], as_index=False)['checkouts'].sum()
    categories = pd.Index(sorted(df[category_column].unique()), name=category_column)
    full_grid = pd.MultiIndex.from_product([all_dates, categories])
    padded = aggregated.set_index(['date', category_column]).reindex(full_grid)
    padded['checkouts'] = padded['checkouts'].fillna(0).astype(int)
    return padded.reset_index()


# 1. Monthly checkouts, digital vs physical (48 months x 2 categories = 96 rows)
monthly_by_usageclass = monthly_grid('usageclass')

# 2. Monthly checkouts by material type (48 months x 3 categories = 144 rows)
monthly_by_materialtype = monthly_grid('materialtype')

# 3. Format share of each year's total
format_share_by_year = (
    df.groupby(['checkoutyear', 'materialtype'], as_index=False)['checkouts'].sum()
)
year_totals = format_share_by_year.groupby('checkoutyear')['checkouts'].transform('sum')
format_share_by_year['pct_of_year'] = (
    format_share_by_year['checkouts'] / year_totals * 100
).round(1)

# 4. Top 100 titles on the normalized key, with each title's leading format
title_totals = df.groupby('clean_title')['checkouts'].sum().sort_values(ascending=False)
top_titles = title_totals.head(100)
primary_materialtype = (
    df[df['clean_title'].isin(top_titles.index)]
      .groupby(['clean_title', 'materialtype'])['checkouts'].sum()
      .sort_values(ascending=False)
      .reset_index()
      .drop_duplicates('clean_title')      # highest-checkout format per title
      .set_index('clean_title')['materialtype']
)
top_100_titles = (
    top_titles.rename('total_checkouts').reset_index()
              .assign(primary_materialtype=lambda d: d['clean_title'].map(primary_materialtype))
)
top_100_titles.insert(0, 'rank', range(1, len(top_100_titles) + 1))
top_100_titles = top_100_titles[
    ['rank', 'clean_title', 'primary_materialtype', 'total_checkouts']
]

# 5. Top 50 creators on the normalized key (null creators cannot be ranked)
top_50_creators = (
    df.dropna(subset=['clean_creator'])
      .groupby('clean_creator')
      .agg(total_checkouts=('checkouts', 'sum'),
           unique_titles=('clean_title', 'nunique'))
      .sort_values('total_checkouts', ascending=False)
      .head(50)
      .reset_index()
)
top_50_creators.insert(0, 'rank', range(1, len(top_50_creators) + 1))

exports = {
    'monthly_checkouts_by_usageclass.csv': monthly_by_usageclass,
    'monthly_checkouts_by_materialtype.csv': monthly_by_materialtype,
    'format_share_by_year.csv': format_share_by_year,
    'top_100_titles.csv': top_100_titles,
    'top_50_creators.csv': top_50_creators,
}

print(f"Writing {len(exports)} files to {OUTPUT_DIR.resolve()}\n")
print(f"{'file':<40} {'rows':>7}  {'size':>10}")
print("-" * 60)
for filename, frame in exports.items():
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"{filename:<40} {len(frame):>7,}  {path.stat().st_size / 1e3:>7.1f} KB")

# --- Sanity checks --------------------------------------------------------
usageclass_total = monthly_by_usageclass['checkouts'].sum()
materialtype_total = monthly_by_materialtype['checkouts'].sum()
pct_by_year = format_share_by_year.groupby('checkoutyear')['pct_of_year'].sum()
pct_within_tolerance = pct_by_year.sub(100).abs().le(0.5)

# Padding must produce complete grids, and must not invent checkouts
assert len(monthly_by_usageclass) == EXPECTED_MONTHS * 2, "usageclass grid is not 96 rows"
assert len(monthly_by_materialtype) == EXPECTED_MONTHS * 3, "materialtype grid is not 144 rows"

print("\nSanity checks")
print("-" * 60)
print(f"usageclass total   {usageclass_total:>12,}  vs expected {EXPECTED_TOTAL:,}  "
      f"{'OK' if usageclass_total == EXPECTED_TOTAL else 'MISMATCH'}")
print(f"materialtype total {materialtype_total:>12,}  vs expected {EXPECTED_TOTAL:,}  "
      f"{'OK' if materialtype_total == EXPECTED_TOTAL else 'MISMATCH'}")
print(f"usageclass grid    {len(monthly_by_usageclass):>12,}  rows, expected {EXPECTED_MONTHS * 2}  "
      f"({(monthly_by_usageclass['checkouts'] == 0).sum()} padded to zero)")
print(f"materialtype grid  {len(monthly_by_materialtype):>12,}  rows, expected {EXPECTED_MONTHS * 3}  "
      f"({(monthly_by_materialtype['checkouts'] == 0).sum()} padded to zero)")
print("\npct_of_year per year (100 +/- 0.5 after rounding):")
for year, pct in pct_by_year.items():
    print(f"  {year}  {pct:6.1f}  {'OK' if pct_within_tolerance[year] else 'OUT OF RANGE'}")

Writing 5 files to D:\Library Analytics\Output

file                                        rows        size
------------------------------------------------------------
monthly_checkouts_by_usageclass.csv           96      2.6 KB
monthly_checkouts_by_materialtype.csv        144      3.7 KB
format_share_by_year.csv                      12      0.4 KB
top_100_titles.csv                           100      4.7 KB
top_50_creators.csv                           50      1.4 KB

Sanity checks
------------------------------------------------------------
usageclass total     16,451,540  vs expected 16,451,540  OK
materialtype total   16,451,540  vs expected 16,451,540  OK
usageclass grid              96  rows, expected 96  (4 padded to zero)
materialtype grid           144  rows, expected 144  (4 padded to zero)

pct_of_year per year (100 +/- 0.5 after rounding):
  2020   100.0  OK
  2021   100.1  OK
  2022   100.0  OK
  2023   100.0  OK


In [40]:
# Step 5c: Draft findings.md
#
# Writes the project write-up to ../findings.md. Figures that depend on title
# and creator normalization are recomputed here from the files Steps 5a and 5b
# just wrote, so the document follows any change to Step 4b:
#   - #1 and #2 titles and their totals          (Output/top_100_titles.csv)
#   - title and creator merge rates              (Data/Clean, raw vs cleaned)
#   - Patterson / Reid volumes, titles and ratio (Output/top_50_creators.csv)
#   - 2023 audiobook vs ebook shares             (Output/format_share_by_year.csv)
#   - the top-50-excluded margin from Q7         (Data/Clean, 2023 rows)
# Figures that do not depend on normalization (monthly and yearly volumes,
# the closure months, digital share) were established in Steps 3-3b and are
# held as literal text.
#
# Author figures are the post-normalization ones, so checkout totals and title
# counts come from the same pass. Re-running this cell overwrites the file.

import pandas as pd
from pathlib import Path

FINDINGS_PATH = Path('../findings.md')
CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
OUTPUT_DIR = Path('../Output')
SMALL_WORDS = {'a', 'an', 'and', 'the', 'of', 'in', 'on', 'to', 'for', 'at', 'by', 'or'}
EXCLUDE_TOP = 50   # per-format exclusion used in the Q7 robustness check

clean = pd.read_csv(CLEAN_PATH, dtype={'isbn': str},
                    usecols=['title', 'creator', 'clean_title', 'clean_creator',
                             'checkoutyear', 'materialtype', 'checkouts'])
assert clean['checkouts'].sum() == 16_451_540
top_titles = pd.read_csv(OUTPUT_DIR / 'top_100_titles.csv')
top_creators = pd.read_csv(OUTPUT_DIR / 'top_50_creators.csv').set_index('clean_creator')
format_share = pd.read_csv(OUTPUT_DIR / 'format_share_by_year.csv')


def display_title(clean_title):
    """'braiding sweetgrass: indigenous ...' -> 'Braiding Sweetgrass'."""
    words = clean_title.split(':')[0].split()
    return ' '.join(w if (i and w in SMALL_WORDS) else w[:1].upper() + w[1:]
                    for i, w in enumerate(words))


# Merge rates
title_merge = (1 - clean['clean_title'].nunique() / clean['title'].nunique()) * 100
creator_merge = (1 - clean['clean_creator'].nunique() / clean['creator'].nunique()) * 100

# #1 and #2 titles, plus any print record filed under the bare (subtitle-less) title
title_totals = clean.groupby('clean_title')['checkouts'].sum()
first, second = top_titles.iloc[0], top_titles.iloc[1]
lead_pct = (first['total_checkouts'] / second['total_checkouts'] - 1) * 100


def bare_remainder(key):
    """Checkouts on the subtitle-less key of a subtitled title, if it exists separately."""
    base = key.split(':')[0].strip()
    return int(title_totals.get(base, 0)) if base != key else 0


second_bare = bare_remainder(second['clean_title'])
first_bare = bare_remainder(first['clean_title'])
title_bullet = (
    f"- **{display_title(first['clean_title'])} is the most-borrowed title at "
    f"{first['total_checkouts']:,} checkouts**: it finishes {lead_pct:.1f}% ahead of "
    f"{display_title(second['clean_title'])} at {second['total_checkouts']:,}"
)
if second['total_checkouts'] + second_bare > first['total_checkouts'] + first_bare:
    title_bullet += (
        f". {display_title(second['clean_title'])}'s print edition is catalogued without its "
        f"subtitle and counted separately ({second_bare:,} checkouts); added back, it would "
        f"lead, so the top of the list is a tie in practice."
    )
elif lead_pct < 5:
    title_bullet += ", making the top of the list a tie in practice."
else:
    title_bullet += "."

# Patterson and Reid
patterson = top_creators.loc['James Patterson']
reid = top_creators.loc['Taylor Jenkins Reid']
breadth_ratio = patterson['unique_titles'] / reid['unique_titles']
volume_gap = abs(patterson['total_checkouts'] / reid['total_checkouts'] - 1) * 100
volume_word = 'the same' if volume_gap < 2 else 'similar'
author_bullet = (
    f"- **James Patterson and Taylor Jenkins Reid draw {volume_word} volume from catalogs "
    f"{breadth_ratio:.0f} times apart**: Patterson takes {patterson['total_checkouts']:,} "
    f"checkouts across {patterson['unique_titles']} titles against Reid's "
    f"{reid['total_checkouts']:,} across {reid['unique_titles']}, a {volume_gap:.1f}% difference "
    f"in volume — catalog breadth and per-title velocity arriving at the same place by "
    f"opposite routes."
)

# 2023 audiobook vs ebook, with the Q7 top-50 exclusion
share_2023 = format_share.query('checkoutyear == 2023').set_index('materialtype')['pct_of_year']
audio_pct, ebook_pct = share_2023['AUDIOBOOK'], share_2023['EBOOK']
by_title_2023 = (clean.query('checkoutyear == 2023')
                      .groupby(['materialtype', 'clean_title'])['checkouts'].sum())
top_each = by_title_2023.sort_values(ascending=False).groupby(level='materialtype').head(EXCLUDE_TOP)
remaining = by_title_2023.drop(top_each.index).groupby(level='materialtype').sum()
excl_margin = int(remaining['EBOOK'] - remaining['AUDIOBOOK'])
excl_leader = 'ebooks' if excl_margin > 0 else 'audiobooks'
format_bullet = (
    f"- **Audiobooks matched ebooks as the largest format in 2023, within "
    f"{abs(audio_pct - ebook_pct):.1f} percentage points**: {audio_pct:.1f}% against "
    f"{ebook_pct:.1f}%, with {excl_leader} ahead by ~{abs(excl_margin):,} checkouts once each "
    f"format's top {EXCLUDE_TOP} titles are excluded (robustness check in Q7), after "
    f"audiobooks sat near 31% for three straight years."
)

merge_bullet = (
    f"- **Normalizing titles and creators collapsed {title_merge:.0f}% of title variants and "
    f"{creator_merge:.0f}% of creator variants, revealing the true rankings**: the physical and "
    f"digital catalogs spell the same work differently, so pre-normalization rankings split a "
    f"single book across three entries and understated its real demand."
)

FINDINGS = f'''# Seattle Public Library — Checkouts Analytics (2020–2023)

## Overview

This analysis covers 16,451,540 checkouts recorded by Seattle Public Library across 48 consecutive months, January 2020 through December 2023. The source is the library's "Checkouts by Title" dataset, pulled from the Socrata open-data API and filtered to physical books, ebooks and audiobooks that reached at least five checkouts in a given month, leaving 1,342,223 title-month records. The window brackets the pandemic branch closures and the three years of recovery that followed, so the data captures both the collapse of physical circulation and the format shift that outlasted it: audiobooks drawing level with ebooks, digital holding 70% of volume long after the branches reopened, and total circulation growing every year. Built as part of a data analytics portfolio; the notebook, cleaned dataset and Power BI aggregations are listed at the end.

## Key Findings

- **Physical circulation produced no qualifying titles for four consecutive months, April through July 2020**: not one physical title reached five checkouts in a month during the closures, and volume resumed at just 641 in August 2020 before reaching 18,310 that September.
- **Digital still accounted for 70.2% of checkouts in 2023**: three years after the branches reopened, the pandemic shift held — digital came off its 81.4% peak in 2020 but never returned to a print-led split.
{format_bullet}
- **Total checkouts grew every year, from 3.28M in 2020 to 4.96M in 2023**: circulation rose 51% across the period, with each year higher than the last despite the closure year at the start.
- **Ebook share fell from 49.7% to 34.8% over four years**: ebooks carried the 2020 closure alone, then gave share back to both audiobooks and returning print.
{merge_bullet}
{author_bullet}
{title_bullet}

## Methodology Notes

- Source: Seattle Public Library "Checkouts by Title", retrieved from the Socrata open-data API at data.seattle.gov.
- Scope: checkout years 2020–2023, restricted to the three main material types — BOOK, EBOOK and AUDIOBOOK.
- Threshold: records below five checkouts in a month were excluded at download, which keeps the extract tractable but means a zero in these tables says no title cleared that floor, not that nothing circulated.
- Grain: one row per title, edition, material type, checkout type, year and month. `checkouts` is a monthly sum, so every volume figure sums that column; row counts are never used as a stand-in. Rows are not deduplicated, since repeated titles are separate editions.
- Normalization: titles have the print catalog's spaced subtitle colon (" : ") and repeated whitespace collapsed, are lowercased, and lose "(unabridged)", bracketed format markers, trailing " / Author." catalog suffixes, series suffixes such as ": … series, book 1", and stock subtitles (": a novel", ": a memoir", ": a story", ": a true story", ": the unabridged edition"). Creators are converted from "Surname, Forename, 1966-" to "Forename Surname", with publisher and placeholder names (DK, Disney, Various, Anonymous) blanked. Title and author rankings quote post-normalization figures throughout.
- Limitation: subtitle stripping covers a fixed list of stock phrases, so a print record catalogued without a subtitle that its digital edition carries (bare "Braiding Sweetgrass" against "Braiding Sweetgrass: Indigenous Wisdom, …") still counts separately, and two different works sharing a cleaned title still merge.
- Limitation: creator title-casing flattens internal capitals, rendering "McDonald" as "Mcdonald".

## Files

- `Notebooks/01_eda.ipynb` — full exploratory analysis: API pull, data quality checks, monthly trends, the COVID gap investigation, rankings, and title/creator normalization.
- `Data/Clean/seattle_checkouts_2020_2023_clean.csv` — the filtered dataset with `date`, `clean_title` and `clean_creator` added alongside the original columns.
- `Output/monthly_checkouts_by_usageclass.csv` — monthly checkouts, digital versus physical.
- `Output/monthly_checkouts_by_materialtype.csv` — monthly checkouts by material type.
- `Output/format_share_by_year.csv` — yearly checkouts and percentage share per material type.
- `Output/top_100_titles.csv` — top 100 titles on the normalized key, with each title's primary format.
- `Output/top_50_creators.csv` — top 50 creators on the normalized key, with distinct title counts.
'''

FINDINGS_PATH.write_text(FINDINGS, encoding='utf-8')

print(f"Wrote:  {FINDINGS_PATH.resolve()}")
print(f"Exists: {FINDINGS_PATH.exists()}")
print(f"Size:   {FINDINGS_PATH.stat().st_size / 1e3:.1f} KB")
print(f"Lines:  {len(FINDINGS.splitlines())}")
print(f"Bullets under Key Findings: {FINDINGS.split('## Methodology')[0].count(chr(10) + '- **')}")
print("\nRecomputed figures:")
print(f"  #1 {first['clean_title']!r} {first['total_checkouts']:,}; "
      f"#2 {second['clean_title']!r} {second['total_checkouts']:,} (lead {lead_pct:.1f}%)")
print(f"  bare-title remainder: #1 {first_bare:,}, #2 {second_bare:,}")
print(f"  merge: titles {title_merge:.1f}%, creators {creator_merge:.1f}%")
print(f"  Patterson {patterson['total_checkouts']:,} / {patterson['unique_titles']} titles; "
      f"Reid {reid['total_checkouts']:,} / {reid['unique_titles']} titles; "
      f"ratio {breadth_ratio:.1f}x, volume gap {volume_gap:.1f}%")
print(f"  2023 audiobook {audio_pct}% vs ebook {ebook_pct}%; top-{EXCLUDE_TOP} excluded margin "
      f"{excl_leader} +{abs(excl_margin):,}")

Wrote:  D:\Library Analytics\findings.md
Exists: True
Size:   5.6 KB
Lines:  36
Bullets under Key Findings: 8

Recomputed figures:
  #1 'where the crawdads sing' 27,896; #2 'braiding sweetgrass: indigenous wisdom, scientific knowledge and the teachings of plants' 27,214 (lead 2.5%)
  bare-title remainder: #1 0, #2 1,496
  merge: titles 41.8%, creators 25.4%
  Patterson 52,775 / 338 titles; Reid 52,505 / 10 titles; ratio 33.8x, volume gap 0.5%
  2023 audiobook 35.4% vs ebook 34.8%; top-50 excluded margin ebooks +1,769


In [41]:
# Q1: Physical checkout recovery vs Jan 2020 baseline
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Physical volume is the sum of `checkouts` per month, never a row count, padded
# onto the full 48-month range so the April-July 2020 closure shows as 0 rather
# than dropping out of the table.
#
# Thresholds are searched only from the first month after the closure (the month
# following the last zero), so pre-closure months like Feb-Mar 2020 cannot count
# as "recovery" and Jan 2020 cannot trivially hit 100% of itself.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
assert df['checkouts'].sum() == 16_451_540   # Step 3 total

all_months = pd.date_range('2020-01-01', '2023-12-01', freq='MS', name='date')

physical = (
    df[df['usageclass'] == 'Physical']
    .groupby('date')['checkouts'].sum()
    .reindex(all_months, fill_value=0)
    .astype(int)
    .rename('physical_checkouts')
    .reset_index()
)
assert len(physical) == 48

# 1. Baseline: January 2020 Physical total
baseline = int(physical.loc[physical['date'] == '2020-01-01', 'physical_checkouts'].iloc[0])
print(f"Jan 2020 Physical baseline: {baseline:,}")

# 2. Every month as a percentage of the baseline
physical['pct_of_baseline'] = (physical['physical_checkouts'] / baseline * 100).round(1)

# 3. First month reaching each threshold after the closure
last_zero = physical.loc[physical['physical_checkouts'] == 0, 'date'].max()
recovery = physical[physical['date'] > last_zero]
print(f"Recovery window starts: {recovery['date'].iloc[0]:%Y-%m} (month after last zero, {last_zero:%Y-%m})")

print("\nFirst month reaching each share of baseline:")
for threshold in (50, 75, 90, 100):
    hits = recovery[recovery['pct_of_baseline'] >= threshold]
    if hits.empty:
        print(f"  {threshold:>3}%: NEVER REACHED")
    else:
        first = hits.iloc[0]
        print(f"  {threshold:>3}%: {first['date']:%Y-%m}  "
              f"({first['physical_checkouts']:,} checkouts, {first['pct_of_baseline']}%)")

# 4. Peak Physical month, 2021-2023
later = physical[physical['date'].dt.year.between(2021, 2023)]
peak = later.loc[later['physical_checkouts'].idxmax()]
print(f"\nPeak Physical month 2021-2023: {peak['date']:%Y-%m}  "
      f"({peak['physical_checkouts']:,} checkouts, {peak['pct_of_baseline']}% of baseline)")

# 5. Full 48-month table
print("\nPhysical checkouts by month vs Jan 2020 baseline:")
print(physical.assign(date=physical['date'].dt.strftime('%Y-%m')).to_string(index=False))

Jan 2020 Physical baseline: 202,835
Recovery window starts: 2020-08 (month after last zero, 2020-07)

First month reaching each share of baseline:
   50%: 2021-07  (117,588 checkouts, 58.0%)
   75%: NEVER REACHED
   90%: NEVER REACHED
  100%: NEVER REACHED

Peak Physical month 2021-2023: 2023-01  (144,997 checkouts, 71.5% of baseline)

Physical checkouts by month vs Jan 2020 baseline:
   date  physical_checkouts  pct_of_baseline
2020-01              202835            100.0
2020-02              180910             89.2
2020-03               94858             46.8
2020-04                   0              0.0
2020-05                   0              0.0
2020-06                   0              0.0
2020-07                   0              0.0
2020-08                 641              0.3
2020-09               18310              9.0
2020-10               32040             15.8
2020-11               35237             17.4
2020-12               44840             22.1
2021-01               54453

In [42]:
# Q2: Post-reopening digital share trajectory
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Digital share is Digital checkouts over all checkouts for the month, both
# summed from `checkouts`, never counted as rows. The window opens in Aug 2020,
# the first month Physical circulation resumed after the closure (see Q1).
#
# The series is V-shaped, so one straight line through all 41 months mislabels
# it. Two least-squares lines are fitted on unrounded shares instead:
#   phase 1, 2020-08 to 2021-12 (branches reopening, print returning)
#   phase 2, 2022-01 to 2023-12 (after the physical recovery ramp)
# Each slope reads as percentage points of share per month; under 0.1 points a
# month either way is treated as stable.
#
# The Aug 2020 "peak" is a startup artifact: Physical had just restarted with
# 641 checkouts, so digital was ~100% of a month that barely had print at all.

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
STABLE_SLOPE = 0.1   # pct points per month
PHASES = {
    'phase 1 (2020-08 to 2021-12)': ('2020-08-01', '2021-12-01'),
    'phase 2 (2022-01 to 2023-12)': ('2022-01-01', '2023-12-01'),
}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

window = pd.date_range('2020-08-01', '2023-12-01', freq='MS', name='date')

# 1. Monthly Digital share of total checkouts (unrounded; rounded only for display)
monthly = (
    df.pivot_table(index='date', columns='usageclass', values='checkouts',
                   aggfunc='sum', fill_value=0)
    .reindex(window, fill_value=0)
)
share = pd.DataFrame({
    'digital_checkouts': monthly['Digital'].astype(int),
    'total_checkouts': monthly[['Digital', 'Physical']].sum(axis=1).astype(int),
})
share['digital_share_pct'] = share['digital_checkouts'] / share['total_checkouts'] * 100
share = share.reset_index()
assert len(share) == 41


def trend_label(slope):
    if abs(slope) < STABLE_SLOPE:
        return 'stable'
    return 'declining' if slope < 0 else 'growing'


# 2. Separate linear trends for each phase, slope in pct points per month
print(f"Aug 2020 digital share: {share['digital_share_pct'].iloc[0]:.1f}%  "
      f"(startup artifact: Physical had just restarted at 641 checkouts)")
print(f"Dec 2023 digital share: {share['digital_share_pct'].iloc[-1]:.1f}%\n")

for label, (start, end) in PHASES.items():
    phase = share[share['date'].between(start, end)]
    slope, _ = np.polyfit(np.arange(len(phase)), phase['digital_share_pct'], 1)
    print(f"Trend {label}: {slope:+.3f} pct points/month over {len(phase)} months  "
          f"->  {trend_label(slope)}")

# 3. Inflection: the low point of the V
trough = share.loc[share['digital_share_pct'].idxmin()]
print(f"\nInflection: digital share bottomed at {trough['date']:%Y-%m} "
      f"({trough['digital_share_pct']:.1f}%); phases split at 2022-01")

# 4. Peak Digital share month in the window
peak = share.loc[share['digital_share_pct'].idxmax()]
print(f"Peak digital share: {peak['date']:%Y-%m}  ({peak['digital_share_pct']:.1f}%) "
      f"-- startup artifact, not a trend high")

# 5. Full 41-month table
print("\nDigital share of monthly checkouts, Aug 2020 - Dec 2023:")
print(share.assign(date=share['date'].dt.strftime('%Y-%m'))
           .to_string(index=False, formatters={'digital_share_pct': '{:.1f}'.format}))

Aug 2020 digital share: 99.7%  (startup artifact: Physical had just restarted at 641 checkouts)
Dec 2023 digital share: 75.4%

Trend phase 1 (2020-08 to 2021-12): -2.071 pct points/month over 17 months  ->  declining
Trend phase 2 (2022-01 to 2023-12): +0.409 pct points/month over 24 months  ->  growing

Inflection: digital share bottomed at 2022-04 (61.8%); phases split at 2022-01
Peak digital share: 2020-08  (99.7%) -- startup artifact, not a trend high

Digital share of monthly checkouts, Aug 2020 - Dec 2023:
   date  digital_checkouts  total_checkouts digital_share_pct
2020-08             245672           246313              99.7
2020-09             227931           246241              92.6
2020-10             225511           257551              87.6
2020-11             223094           258331              86.4
2020-12             241753           286593              84.4
2021-01             254685           309138              82.4
2021-02             217088           270763     

In [43]:
# Q3: Cultural-moment titles — spike factor analysis
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Titles are keyed on `clean_title` from Step 4b, so a work's print, ebook and
# audiobook editions count together. Volume is the sum of `checkouts`.
#
# Each top-100 title gets a full 48-month series, with months it never cleared
# the five-checkout floor filled as 0. spike_factor compares its best month to
# its typical month (the median); when the median is 0 the mean stands in.
# Note the 48-month window includes months before a title was published, so
# recent releases carry pre-release zeros that pull their median down.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 100
SPIKE_THRESHOLD = 5

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

all_months = pd.date_range('2020-01-01', '2023-12-01', freq='MS', name='date')

top_titles = (
    df.groupby('clean_title')['checkouts'].sum()
    .nlargest(TOP_N)
    .index
)

# 1. Monthly checkouts per title, one column per month across all 48
monthly = (
    df[df['clean_title'].isin(top_titles)]
    .pivot_table(index='clean_title', columns='date', values='checkouts',
                 aggfunc='sum', fill_value=0)
    .reindex(columns=all_months, fill_value=0)
)
assert monthly.shape == (TOP_N, 48)

# 2-3. Peak, median and spike factor per title
spikes = pd.DataFrame({
    'peak_month': monthly.idxmax(axis=1).dt.strftime('%Y-%m'),
    'peak_checkouts': monthly.max(axis=1),
    'median_monthly_checkouts': monthly.median(axis=1),
})
baseline = spikes['median_monthly_checkouts'].where(
    spikes['median_monthly_checkouts'] > 0, monthly.mean(axis=1)
)
spikes['spike_factor'] = (spikes['peak_checkouts'] / baseline).round(2)
spikes = spikes.reset_index()

# 4-5. Flagged titles, sharpest spike first
flagged = (
    spikes[spikes['spike_factor'] >= SPIKE_THRESHOLD]
    .sort_values('spike_factor', ascending=False)
)
print(f"Titles with spike_factor >= {SPIKE_THRESHOLD}: {len(flagged)} of {TOP_N}")
print(f"Median of 0 replaced by the mean for: {(spikes['median_monthly_checkouts'] == 0).sum()} titles\n")
print(flagged[['clean_title', 'peak_month', 'peak_checkouts',
               'median_monthly_checkouts', 'spike_factor']].to_string(index=False))

# 6. Do the spikes cluster in particular months?
print("\nPeak months of flagged titles:")
print(
    flagged.groupby('peak_month').size()
    .rename('flagged_titles')
    .sort_index()
    .to_string()
)

Titles with spike_factor >= 5: 25 of 100
Median of 0 replaced by the mean for: 10 titles

                                                             clean_title peak_month  peak_checkouts  median_monthly_checkouts  spike_factor
                                                            the swimmers    2023-04            4239                       3.0       1413.00
                                              the house of broken angels    2022-09            4180                      63.0         66.35
                                          so you want to talk about race    2020-06            5218                     102.5         50.91
 white fragility: why it's so hard for white people to talk about racism    2020-06            3211                      88.5         36.28
                                                     the paris apartment    2023-01             663                      21.0         31.57
                                                         a promised la

In [44]:
# Q4: Pareto distribution of checkouts across titles
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Titles are keyed on `clean_title` from Step 4b; volume is the sum of
# `checkouts`. Ties are broken alphabetically so the ranking is repeatable.
#
# Caveat: the extract drops any title-month under five checkouts, so the long
# tail of rarely borrowed titles is mostly missing. Concentration measured here
# is concentration among titles that cleared that floor at least once, and the
# Gini below understates the collection-wide figure.

import math

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_SHARES = (0.1, 1, 5, 10, 25, 50)   # percent of titles

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

# 1. Total checkouts per title, largest first
titles = (
    df.groupby('clean_title', as_index=False)['checkouts'].sum()
    .sort_values(['checkouts', 'clean_title'], ascending=[False, True])
    .reset_index(drop=True)
)

# 2. Cumulative checkouts and cumulative share of titles
n_titles = len(titles)
total_checkouts = int(titles['checkouts'].sum())
assert total_checkouts == 16_451_540   # Step 3 total
titles['cumulative_checkouts'] = titles['checkouts'].cumsum()
titles['cumulative_title_share'] = (np.arange(1, n_titles + 1) / n_titles * 100)

# 3. Totals
print(f"Unique clean_titles: {n_titles:,}")
print(f"Total checkouts:     {total_checkouts:,}")

# 4. Share of checkouts held by the top slices of titles
rows = []
for pct in TOP_SHARES:
    count = math.ceil(n_titles * pct / 100)
    cumulative = int(titles['cumulative_checkouts'].iloc[count - 1])
    rows.append({
        'top_pct': f"{pct}%",
        'title_count': count,
        'pct_of_titles': round(count / n_titles * 100, 2),
        'cumulative_checkouts': cumulative,
        'pct_of_total_checkouts': round(cumulative / total_checkouts * 100, 1),
    })
print("\nCheckout concentration:")
print(pd.DataFrame(rows).to_string(index=False))

# 5. Gini coefficient over per-title totals (0 = perfectly even, 1 = one title takes all)
ascending = np.sort(titles['checkouts'].to_numpy())
ranks = np.arange(1, n_titles + 1)
gini = (2 * np.sum(ranks * ascending)) / (n_titles * ascending.sum()) - (n_titles + 1) / n_titles
print(f"\nGini coefficient: {gini:.3f}")

# 6. Sanity check: both ends of the ranking
print("\nTop 20 titles:")
print(titles.head(20)[['clean_title', 'checkouts']].to_string(index=False))
print("\nBottom 20 titles:")
print(titles.tail(20)[['clean_title', 'checkouts']].to_string(index=False))

Unique clean_titles: 103,856
Total checkouts:     16,451,540

Checkout concentration:
top_pct  title_count  pct_of_titles  cumulative_checkouts  pct_of_total_checkouts
   0.1%          104            0.1               1292698                     7.9
     1%         1039            1.0               4586780                    27.9
     5%         5193            5.0               9027102                    54.9
    10%        10386           10.0              11292988                    68.6
    25%        25964           25.0              14253018                    86.6
    50%        51928           50.0              15842018                    96.3

Gini coefficient: 0.789

Top 20 titles:
                                                                             clean_title  checkouts
                                                                 where the crawdads sing      27896
braiding sweetgrass: indigenous wisdom, scientific knowledge and the teachings of plants      27214

In [45]:
# Q5: Pareto comparison — Digital vs Physical
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Repeats the Q4 concentration measures within each usage class: a title's
# Digital and Physical checkouts are ranked separately, so a work held in both
# formats appears once in each class. Volume is the sum of `checkouts`.
#
# The five-checkout monthly floor from Q4 applies here too, and it bites harder
# on whichever class has the longer tail of rarely borrowed titles.

import math

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_SHARES = (1, 5, 10)   # percent of titles

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])


def concentration(per_title):
    """Title count, volume, top-slice shares and Gini for one class's per-title totals."""
    ranked = np.sort(per_title.to_numpy())[::-1]
    n_titles = len(ranked)
    total = int(ranked.sum())
    cumulative = ranked.cumsum()

    stats = {'unique_titles': n_titles, 'total_checkouts': total}
    for pct in TOP_SHARES:
        count = math.ceil(n_titles * pct / 100)
        stats[f'top_{pct}pct_share'] = round(cumulative[count - 1] / total * 100, 1)

    ascending = ranked[::-1]
    ranks = np.arange(1, n_titles + 1)
    stats['gini'] = round(
        (2 * np.sum(ranks * ascending)) / (n_titles * total) - (n_titles + 1) / n_titles, 3
    )
    return stats


# 1-4. Per-title totals and concentration measures for each class
comparison = pd.DataFrame({
    usage: concentration(group.groupby('clean_title')['checkouts'].sum())
    for usage, group in df.groupby('usageclass')
})[['Digital', 'Physical']]

assert comparison.loc['total_checkouts'].sum() == 16_451_540   # Step 3 total

print("Checkout concentration by usage class:")
display_formats = {
    'unique_titles': '{:,.0f}',
    'total_checkouts': '{:,.0f}',
    **{f'top_{pct}pct_share': '{:.1f}%' for pct in TOP_SHARES},
    'gini': '{:.3f}',
}
print(comparison.apply(
    lambda row: row.map(display_formats[row.name].format), axis=1
).to_string())

Checkout concentration by usage class:
                    Digital   Physical
unique_titles        69,763     56,851
total_checkouts  11,764,523  4,687,017
top_1pct_share        27.0%      19.8%
top_5pct_share        54.7%      43.9%
top_10pct_share       69.1%      59.2%
gini                  0.794      0.722


In [46]:
# Q6: Catalog breadth vs total checkouts — prolific authors
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Authors are keyed on `clean_creator` and titles on `clean_title`, both from
# Step 4b, so an author's print, ebook and audiobook editions of one work count
# as a single title. Rows with no creator are dropped. Volume is the sum of
# `checkouts`.
#
# Pearson measures a straight-line relationship and is pulled around by a few
# very large catalogs; Spearman compares ranks only, so it shows whether broader
# catalogs reliably mean more checkouts regardless of scale.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
MIN_TITLES = 50

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

# 1. Catalog breadth and volume per author
authors = (
    df.dropna(subset=['clean_creator'])
    .groupby('clean_creator')
    .agg(unique_titles=('clean_title', 'nunique'), total_checkouts=('checkouts', 'sum'))
)

# 2. Prolific authors only
prolific = authors[authors['unique_titles'] >= MIN_TITLES].copy()

# 3. Does a broader catalog mean more checkouts?
pearson = prolific['unique_titles'].corr(prolific['total_checkouts'], method='pearson')
spearman = prolific['unique_titles'].corr(prolific['total_checkouts'], method='spearman')

# 4. Average pull of each title in the catalog
prolific['mean_checkouts_per_title'] = (
    prolific['total_checkouts'] / prolific['unique_titles']
).round(1)

# 5. Results
print(f"Authors with >= {MIN_TITLES} unique titles: {len(prolific):,} "
      f"(of {len(authors):,} authors overall)")
print(f"Pearson  r   (unique_titles vs total_checkouts): {pearson:.3f}")
print(f"Spearman rho (unique_titles vs total_checkouts): {spearman:.3f}")

ranked = prolific.sort_values('mean_checkouts_per_title', ascending=False).reset_index()
columns = ['clean_creator', 'unique_titles', 'total_checkouts', 'mean_checkouts_per_title']

print("\nLabels:")
print("  efficient     = highest mean checkouts per title: each title in the catalog draws heavily")
print("  deep backlist = lowest mean checkouts per title: long-running or classic catalogs whose")
print("                  many titles each circulate lightly (often series or late authors)")

print("\nTop 10 by mean checkouts per title (efficient):")
print(ranked.head(10)[columns].to_string(index=False))
print("\nBottom 10 by mean checkouts per title (deep backlist authors):")
print(ranked.tail(10)[columns].to_string(index=False))

Authors with >= 50 unique titles: 119 (of 41,239 authors overall)
Pearson  r   (unique_titles vs total_checkouts): 0.339
Spearman rho (unique_titles vs total_checkouts): 0.414

Labels:
  efficient     = highest mean checkouts per title: each title in the catalog draws heavily
  deep backlist = lowest mean checkouts per title: long-running or classic catalogs whose
                  many titles each circulate lightly (often series or late authors)

Top 10 by mean checkouts per title (efficient):
   clean_creator  unique_titles  total_checkouts  mean_checkouts_per_title
   J. K. Rowling             50            63530                    1270.6
     Jeff Kinney             54            64474                    1194.0
Michael Connelly             58            60071                    1035.7
      Mo Willems             97            85525                     881.7
  David Baldacci             66            57949                     878.0
    John Grisham             67            51827  

In [47]:
# Q7: Audiobook vs ebook in 2023 — robustness check
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# The 2023 format shares put audiobooks 0.6 points ahead of ebooks. This checks
# whether that lead is broad-based or carried by a handful of hits.
#
# Exclusion is per format: each format loses the checkouts of its own top-N
# titles (on `clean_title`) and nothing else, so a bestseller dropped from the
# audiobook total still counts toward ebooks unless it is top-N there too.
# Shares are then recomputed over whatever checkouts remain. Volume is the sum
# of `checkouts`.
#
# A margin under TIE_PCT_POINTS share points or TIE_CHECKOUTS checkouts is
# reported as a TIE rather than a winner. Margins use unrounded shares.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
FORMATS = ['BOOK', 'EBOOK', 'AUDIOBOOK']
EXCLUSION_SIZES = (20, 50)
TIE_PCT_POINTS = 1.0
TIE_CHECKOUTS = 5_000

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
df_2023 = df[df['checkoutyear'] == 2023]
assert df_2023['checkouts'].sum() == 4_960_175   # Step 3 per-year total

title_totals = df_2023.groupby(['materialtype', 'clean_title'])['checkouts'].sum()


def format_shares(totals):
    """Checkouts and percentage share per format, largest first."""
    shares = totals.reindex(FORMATS, fill_value=0).rename('checkouts').to_frame()
    shares['pct'] = shares['checkouts'] / shares['checkouts'].sum() * 100
    return shares.sort_values('checkouts', ascending=False)


def print_ranking(label, shares):
    """Ranking plus the audiobook-minus-ebook margin; returns the verdict."""
    gap_pct = shares.loc['AUDIOBOOK', 'pct'] - shares.loc['EBOOK', 'pct']
    gap_checkouts = int(shares.loc['AUDIOBOOK', 'checkouts'] - shares.loc['EBOOK', 'checkouts'])
    print(f"\n{label}")
    print(shares.to_string(formatters={'checkouts': '{:,}'.format, 'pct': '{:.1f}%'.format}))
    print(f"  AUDIOBOOK - EBOOK margin: {gap_pct:+.2f} pts, {gap_checkouts:+,} checkouts")
    if abs(gap_pct) < TIE_PCT_POINTS or abs(gap_checkouts) < TIE_CHECKOUTS:
        return 'TIE'
    return 'AUDIOBOOK leads' if gap_checkouts > 0 else 'EBOOK leads'


# 1. Original 2023 totals and shares
original = format_shares(df_2023.groupby('materialtype')['checkouts'].sum())
assert original['checkouts'].sum() == df_2023['checkouts'].sum()
verdicts = {'all titles': print_ranking("2023, all titles:", original)}

for top_n in EXCLUSION_SIZES:
    # 2. Each format's own top-N titles
    top_per_format = (
        title_totals.sort_values(ascending=False)
        .groupby(level='materialtype').head(top_n)
    )

    # 3. Drop those title-format pairs and recompute
    remaining = (
        title_totals.drop(top_per_format.index)
        .groupby(level='materialtype').sum()
    )

    # 4-5. Ranking after exclusion
    removed = top_per_format.groupby(level='materialtype').sum().reindex(FORMATS)
    removed_text = ', '.join(f"{fmt} {removed[fmt]:,}" for fmt in FORMATS)
    verdicts[f'top {top_n} excluded'] = print_ranking(
        f"2023, top {top_n} titles per format excluded (removed: {removed_text}):",
        format_shares(remaining),
    )

# 6. Verdict
print(f"\nAudiobook vs ebook verdict (TIE if within {TIE_PCT_POINTS} pts "
      f"or {TIE_CHECKOUTS:,} checkouts):")
for label, verdict in verdicts.items():
    print(f"  {label:<18} {verdict}")


2023, all titles:
             checkouts   pct
materialtype                
AUDIOBOOK    1,755,879 35.4%
EBOOK        1,728,108 34.8%
BOOK         1,476,188 29.8%
  AUDIOBOOK - EBOOK margin: +0.56 pts, +27,771 checkouts

2023, top 20 titles per format excluded (removed: BOOK 52,000, EBOOK 73,483, AUDIOBOOK 92,037):
             checkouts   pct
materialtype                
AUDIOBOOK    1,663,842 35.1%
EBOOK        1,654,625 34.9%
BOOK         1,424,188 30.0%
  AUDIOBOOK - EBOOK margin: +0.19 pts, +9,217 checkouts

2023, top 50 titles per format excluded (removed: BOOK 98,977, EBOOK 130,094, AUDIOBOOK 159,634):
             checkouts   pct
materialtype                
EBOOK        1,598,014 35.0%
AUDIOBOOK    1,596,245 34.9%
BOOK         1,377,211 30.1%
  AUDIOBOOK - EBOOK margin: -0.04 pts, -1,769 checkouts

Audiobook vs ebook verdict (TIE if within 1.0 pts or 5,000 checkouts):
  all titles         TIE
  top 20 excluded    TIE
  top 50 excluded    TIE


In [48]:
# Q8: Format dominance for multi-format titles
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Titles are keyed on `clean_title` from Step 4b, so a work's print, ebook and
# audiobook editions land in one row with a column per format. Volume is the
# sum of `checkouts`.
#
# Only titles carried in two or more formats with at least 500 checkouts are
# kept. dominant_share is the leading format's slice of the title's total: near
# 100% means readers overwhelmingly picked one format; with three formats the
# floor is about 33%, with two it is 50%.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
FORMATS = ['BOOK', 'EBOOK', 'AUDIOBOOK']
MIN_FORMATS = 2
MIN_CHECKOUTS = 500
LOCKED_SHARE = 70   # pct

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

# 1. Checkouts per format and number of formats per title
by_format = (
    df.pivot_table(index='clean_title', columns='materialtype', values='checkouts',
                   aggfunc='sum', fill_value=0)
    .reindex(columns=FORMATS, fill_value=0)
)
by_format['n_formats'] = (by_format[FORMATS] > 0).sum(axis=1)
by_format['total_checkouts'] = by_format[FORMATS].sum(axis=1)

# 2. Multi-format titles with meaningful volume
multi = by_format[
    (by_format['n_formats'] >= MIN_FORMATS) & (by_format['total_checkouts'] >= MIN_CHECKOUTS)
].copy()

# 3. Leading format and its share of the title's total
multi['dominant_format'] = multi[FORMATS].idxmax(axis=1)
multi['dominant_share'] = (multi[FORMATS].max(axis=1) / multi['total_checkouts'] * 100).round(1)

# 4. How locked-in are titles overall?
print(f"Multi-format titles with >= {MIN_CHECKOUTS} checkouts: {len(multi):,} "
      f"({(multi['n_formats'] == 2).sum():,} in two formats, "
      f"{(multi['n_formats'] == 3).sum():,} in three)")

print("\ndominant_share distribution (%):")
print(multi['dominant_share'].describe(percentiles=[0.25, 0.5, 0.75])
      .loc[['min', '25%', '50%', '75%', 'max']].round(1).to_string())

locked = multi[multi['dominant_share'] >= LOCKED_SHARE]
print(f"\nTitles with dominant_share >= {LOCKED_SHARE}%: {len(locked):,} "
      f"({len(locked) / len(multi) * 100:.1f}%)")
print("Leading format among them:")
print(locked['dominant_format'].value_counts().reindex(FORMATS, fill_value=0).to_string())

# 5. Most locked-in and most balanced titles
columns = ['clean_title', 'n_formats', *FORMATS, 'total_checkouts',
           'dominant_format', 'dominant_share']
ranked = (
    multi.reset_index()
    .sort_values(['dominant_share', 'total_checkouts'], ascending=[False, False])
)
print("\nTop 20: most locked to one format")
print(ranked.head(20)[columns].to_string(index=False))

balanced = multi.reset_index().sort_values(['dominant_share', 'total_checkouts'],
                                           ascending=[True, False])
print("\nBottom 20: most balanced across formats")
print(balanced.head(20)[columns].to_string(index=False))

Multi-format titles with >= 500 checkouts: 5,730 (1,497 in two formats, 4,233 in three)

dominant_share distribution (%):
min    33.6
25%    49.6
50%    56.8
75%    68.9
max    99.6

Titles with dominant_share >= 70%: 1,348 (23.5%)
Leading format among them:
dominant_format
BOOK         472
EBOOK        451
AUDIOBOOK    425

Top 20: most locked to one format
                                                    clean_title  n_formats  BOOK  EBOOK  AUDIOBOOK  total_checkouts dominant_format  dominant_share
                                                           salt          2     5      0       1278             1283       AUDIOBOOK            99.6
                                                         hooked          2     5      0       1007             1012       AUDIOBOOK            99.5
                                                     blood heir          2     5    917          0              922           EBOOK            99.5
                                               

In [49]:
# Q9: Audio-first vs print-first title archetypes
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Titles are keyed on `clean_title` from Step 4b; physical means the BOOK
# material type. Volume is the sum of `checkouts`.
#
# Only titles carried in two or more formats with at least 1,000 checkouts are
# kept. A title is assigned an archetype when one format takes at least 60% of
# its checkouts; the three thresholds cannot overlap, and titles split more
# evenly than that belong to no archetype.
#
# As Q8 found, a stray handful of checkouts from a different work sharing the
# same title can make a single-format title look multi-format. The per-format
# columns are printed so those cases are visible.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
FORMATS = ['BOOK', 'EBOOK', 'AUDIOBOOK']
MIN_FORMATS = 2
MIN_CHECKOUTS = 1_000
FIRST_SHARE = 60   # pct
TOP_N = 15

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

by_format = (
    df.pivot_table(index='clean_title', columns='materialtype', values='checkouts',
                   aggfunc='sum', fill_value=0)
    .reindex(columns=FORMATS, fill_value=0)
)
by_format['total_checkouts'] = by_format[FORMATS].sum(axis=1)
multi = by_format[
    ((by_format[FORMATS] > 0).sum(axis=1) >= MIN_FORMATS)
    & (by_format['total_checkouts'] >= MIN_CHECKOUTS)
].copy()

# 1. Each format's share of the title's checkouts
multi['pct_audiobook'] = (multi['AUDIOBOOK'] / multi['total_checkouts'] * 100).round(1)
multi['pct_ebook'] = (multi['EBOOK'] / multi['total_checkouts'] * 100).round(1)
multi['pct_physical'] = (multi['BOOK'] / multi['total_checkouts'] * 100).round(1)

# 2. Archetypes
archetypes = {
    'Audio-first': multi['pct_audiobook'] >= FIRST_SHARE,
    'Print-first': multi['pct_physical'] >= FIRST_SHARE,
    'Ebook-first': multi['pct_ebook'] >= FIRST_SHARE,
}

print(f"Multi-format titles with >= {MIN_CHECKOUTS:,} checkouts: {len(multi):,}")

# 3. Biggest titles in each archetype
columns = ['clean_title', 'total_checkouts', *FORMATS,
           'pct_audiobook', 'pct_ebook', 'pct_physical']
for name, mask in archetypes.items():
    top = multi[mask].sort_values('total_checkouts', ascending=False).head(TOP_N)
    print(f"\n{name} (>= {FIRST_SHARE}%): top {TOP_N} by total checkouts")
    print(top.reset_index()[columns].to_string(index=False))

# 4. Archetype counts
counts = pd.Series({name: int(mask.sum()) for name, mask in archetypes.items()})
counts['No archetype (mixed)'] = len(multi) - counts.sum()
print("\nTitles per archetype:")
print(counts.to_string())

Multi-format titles with >= 1,000 checkouts: 2,719

Audio-first (>= 60%): top 15 by total checkouts
                                                                             clean_title  total_checkouts  BOOK  EBOOK  AUDIOBOOK  pct_audiobook  pct_ebook  pct_physical
braiding sweetgrass: indigenous wisdom, scientific knowledge and the teachings of plants            27214     0   3472      23742           87.2       12.8           0.0
                                                                                becoming            20511   806   6124      13581           66.2       29.9           3.9
                                                          so you want to talk about race            17452   572   3496      13384           76.7       20.0           3.3
                talking to strangers: what we should know about the people we don't know            16900  1950   4430      10520           62.2       26.2          11.5
                                                  

In [50]:
# Q10: Author archetype clustering (k-means)
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Authors are keyed on `clean_creator` and titles on `clean_title`, both from
# Step 4b. Volume is the sum of `checkouts`; pct_digital and pct_audiobook are
# shares of each author's own checkouts.
#
# Features are standardized before clustering so that unique_titles (tens to
# hundreds) does not outweigh the percentage columns. Total checkouts are logged
# because the top authors differ by an order of magnitude.
#
# Cluster labels are suggested from each cluster's centroid in standardized
# units. The most extreme (cluster, feature) deviations are claimed first, each
# label used at most once; a cluster with no feature at least DISTINCT_SD from
# the top-100 average is the unremarkable middle. The rule runs in code rather
# than being hard-coded per cluster number, so labels follow the profile if the
# data change.

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 100
N_CLUSTERS = 4
DISTINCT_SD = 0.75
FEATURES = ['unique_titles', 'log_total_checkouts', 'mean_checkouts_per_title',
            'pct_digital', 'pct_audiobook']

# Label for a cluster whose most distinctive feature sits well above / below average
LABELS = {
    ('unique_titles', 'high'): 'Catalog Giants',
    ('unique_titles', 'low'): 'Breakout Hits',
    ('log_total_checkouts', 'high'): 'Headliners',
    ('log_total_checkouts', 'low'): 'Steady Midlist',
    ('mean_checkouts_per_title', 'high'): 'Breakout Hits',
    ('mean_checkouts_per_title', 'low'): 'Deep Backlist',
    ('pct_digital', 'high'): 'Digital Natives',
    ('pct_digital', 'low'): 'Print Loyalists',
    ('pct_audiobook', 'high'): 'Audiobook Specialists',
    ('pct_audiobook', 'low'): 'Read-not-Heard',
}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
df = df.dropna(subset=['clean_creator'])

top_creators = df.groupby('clean_creator')['checkouts'].sum().nlargest(TOP_N).index
top = df[df['clean_creator'].isin(top_creators)]

# 1. Feature matrix
grouped = top.groupby('clean_creator')
authors = pd.DataFrame({
    'unique_titles': grouped['clean_title'].nunique(),
    'total_checkouts': grouped['checkouts'].sum(),
})
digital = top[top['usageclass'] == 'Digital'].groupby('clean_creator')['checkouts'].sum()
audiobook = top[top['materialtype'] == 'AUDIOBOOK'].groupby('clean_creator')['checkouts'].sum()

authors['log_total_checkouts'] = np.log(authors['total_checkouts'])
authors['mean_checkouts_per_title'] = authors['total_checkouts'] / authors['unique_titles']
authors['pct_digital'] = digital.reindex(authors.index, fill_value=0) / authors['total_checkouts'] * 100
authors['pct_audiobook'] = audiobook.reindex(authors.index, fill_value=0) / authors['total_checkouts'] * 100
assert len(authors) == TOP_N

# 2. Standardize
scaled = StandardScaler().fit_transform(authors[FEATURES])

# 3-4. Cluster and attach labels
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit(scaled)
authors['cluster'] = kmeans.labels_

# 6. Suggested label per cluster from its most extreme standardized feature
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES)
deviations = centroids.stack().rename('z').reset_index()
deviations.columns = ['cluster', 'feature', 'z']
deviations = deviations.reindex(deviations['z'].abs().sort_values(ascending=False).index)

suggested = {}
for row in deviations.itertuples():
    if abs(row.z) < DISTINCT_SD:
        break
    label = LABELS[(row.feature, 'high' if row.z > 0 else 'low')]
    if row.cluster not in suggested and label not in {l for l, _, _ in suggested.values()}:
        suggested[row.cluster] = (label, row.feature, row.z)
for cluster, centroid in centroids.iterrows():
    if cluster not in suggested:
        feature = centroid.abs().idxmax()
        suggested[cluster] = ('Mainstream Core', feature, centroid[feature])

# 5. Profile of each cluster
profile_format = {
    'unique_titles': '{:.1f}', 'log_total_checkouts': '{:.2f}',
    'mean_checkouts_per_title': '{:,.0f}', 'pct_digital': '{:.1f}%', 'pct_audiobook': '{:.1f}%',
}
print(f"Top {TOP_N} authors in {N_CLUSTERS} clusters\n")
print("Overall mean across all clusters:")
for feature in FEATURES:
    print(f"  {feature:<26} {profile_format[feature].format(authors[feature].mean())}")

for cluster in sorted(authors['cluster'].unique()):
    members = authors[authors['cluster'] == cluster]
    label, feature, z = suggested[cluster]
    print(f"\n=== Cluster {cluster}: {label}  ({len(members)} authors; "
          f"most distinctive: {feature} at {z:+.2f} SD) ===")
    for f in FEATURES:
        print(f"  {f:<26} {profile_format[f].format(members[f].mean())}")
    print(f"  (mean total_checkouts     {members['total_checkouts'].mean():,.0f})")
    print("  Top 5 by total checkouts:")
    for name, row in members.nlargest(5, 'total_checkouts').iterrows():
        print(f"    {name:<28} {row['total_checkouts']:>8,.0f} checkouts, "
              f"{row['unique_titles']:>4.0f} titles, {row['pct_audiobook']:.0f}% audio")

Top 100 authors in 4 clusters

Overall mean across all clusters:
  unique_titles              50.8
  log_total_checkouts        10.35
  mean_checkouts_per_title   2,551
  pct_digital                79.2%
  pct_audiobook              36.6%

=== Cluster 0: Print Loyalists  (15 authors; most distinctive: pct_digital at -2.08 SD) ===
  unique_titles              119.5
  log_total_checkouts        10.32
  mean_checkouts_per_title   362
  pct_digital                31.3%
  pct_audiobook              4.6%
  (mean total_checkouts     34,129)
  Top 5 by total checkouts:
    Mo Willems                     85,525 checkouts,   97 titles, 2% audio
    Dav Pilkey                     79,152 checkouts,  168 titles, 2% audio
    Lincoln Peirce                 50,251 checkouts,  135 titles, 2% audio
    Jim Davis                      29,706 checkouts,  107 titles, 0% audio
    Tracey West                    28,855 checkouts,  125 titles, 11% audio

=== Cluster 1: Mainstream Core  (63 authors; most disti

In [51]:
# Q11: Per-title checkout returns by catalog size bucket
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Authors are keyed on `clean_creator` (rows with no creator dropped) and titles
# on `clean_title`, both from Step 4b. Volume is the sum of `checkouts`.
#
# mean_checkouts_per_title is computed per author, then summarized per bucket:
# the mean is pulled up by a few hits, the median shows the typical author.
# pooled_checkouts_per_title (bucket checkouts / bucket titles) weights each
# title equally rather than each author.
#
# Caveats: the five-checkout floor means an author only appears if at least one
# title cleared it, which flatters the smallest bucket. And as the Q1-Q10 audit
# found, `unique_titles` is inflated for multi-format authors whose print and
# digital editions land on different title keys, which pushes some authors up a
# bucket and understates their per-title figure.

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
BUCKET_EDGES = [0, 10, 50, 200, 500, np.inf]
BUCKET_LABELS = ['1-10', '11-50', '51-200', '201-500', '500+']

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

authors = (
    df.dropna(subset=['clean_creator'])
    .groupby('clean_creator')
    .agg(unique_titles=('clean_title', 'nunique'), total_checkouts=('checkouts', 'sum'))
)
authors['mean_checkouts_per_title'] = authors['total_checkouts'] / authors['unique_titles']

# 1. Catalog size buckets (right-inclusive: 10 titles is '1-10', 501 is '500+')
authors['bucket'] = pd.cut(authors['unique_titles'], bins=BUCKET_EDGES,
                           labels=BUCKET_LABELS, right=True)
assert authors['bucket'].notna().all()

# 2. Per-bucket summary
buckets = authors.groupby('bucket', observed=False).agg(
    authors=('unique_titles', 'size'),
    titles=('unique_titles', 'sum'),
    checkouts=('total_checkouts', 'sum'),
    mean_of_mean_cpt=('mean_checkouts_per_title', 'mean'),
    median_of_mean_cpt=('mean_checkouts_per_title', 'median'),
)
buckets['pooled_checkouts_per_title'] = buckets['checkouts'] / buckets['titles']
buckets['pct_of_checkouts'] = buckets['checkouts'] / buckets['checkouts'].sum() * 100

# 3. Does per-title efficiency fall as catalogs grow?
print(f"Authors: {len(authors):,}   Checkouts with a creator: {int(authors['total_checkouts'].sum()):,}\n")
print("Per-title checkouts by author catalog size (unique clean_titles):")
print(buckets.to_string(formatters={
    'authors': '{:,}'.format,
    'titles': '{:,}'.format,
    'checkouts': '{:,}'.format,
    'mean_of_mean_cpt': '{:,.1f}'.format,
    'median_of_mean_cpt': '{:,.1f}'.format,
    'pooled_checkouts_per_title': '{:,.1f}'.format,
    'pct_of_checkouts': '{:.1f}%'.format,
}))

empty = buckets.index[buckets['authors'] == 0].tolist()
if empty:
    largest = authors['unique_titles'].idxmax()
    print(f"\nEmpty bucket(s): {', '.join(empty)} -- largest catalog is {largest} "
          f"with {authors.loc[largest, 'unique_titles']:,} titles")

medians = buckets.loc[buckets['authors'] > 0, 'median_of_mean_cpt']
direction = 'falls' if medians.is_monotonic_decreasing else (
    'rises' if medians.is_monotonic_increasing else 'is not monotonic')
print(f"Median per-title checkouts {direction} across non-empty buckets, smallest to largest: "
      + ' -> '.join(f"{v:,.1f}" for v in medians))

Authors: 41,239   Checkouts with a creator: 16,307,519

Per-title checkouts by author catalog size (unique clean_titles):
        authors titles checkouts mean_of_mean_cpt median_of_mean_cpt pooled_checkouts_per_title pct_of_checkouts
bucket                                                                                                          
1-10     39,831 73,877 9,507,464            100.5               23.0                      128.7            58.3%
11-50     1,292 25,079 4,813,893            184.9               90.1                      191.9            29.5%
51-200      110  8,396 1,792,758            209.6              130.7                      213.5            11.0%
201-500       6  1,537   193,404            129.1               94.0                      125.8             1.2%
500+          0      0         0              NaN                NaN                        NaN             0.0%

Empty bucket(s): 500+ -- largest catalog is James Patterson with 338 titles
Median per

In [ ]:
# Q12: Content age distribution — 2023 checkouts
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Volume is the sum of `checkouts`; row counts are reported only for how much
# of the data has a usable publication year.
#
# `publicationyear` is free text: "2022", "2022.", "[2022]", "c2017.", "©2008.",
# "[1995, c1967]". pub_year is the first standalone 4-digit number in 1900-2024;
# for two-date strings that is usually the edition year, not the copyright year.
#
# Caveat: this measures the age of the edition borrowed, not of the work. Digital
# records carry the ebook/audiobook release date, so a classic reissued in 2019
# counts as 4 years old. Shares are given against both the usable-age checkouts
# and the full 2023 total, since rows without a usable year are excluded.

import re

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
YEAR_MIN, YEAR_MAX = 1900, 2024
AGE_EDGES = [-0.5, 2, 5, 10, 20, 50, np.inf]   # right-inclusive integer ages
AGE_LABELS = ['0-2 yrs', '3-5 yrs', '6-10 yrs', '11-20 yrs', '21-50 yrs', '50+ yrs']
YEAR_PATTERN = re.compile(r'(?<!\d)(\d{4})(?!\d)')

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str, 'publicationyear': str}, parse_dates=['date'])
df_2023 = df[df['checkoutyear'] == 2023].copy()
total_2023 = int(df_2023['checkouts'].sum())
assert total_2023 == 4_960_175   # Step 3 per-year total


def first_valid_year(text):
    """First standalone 4-digit number within YEAR_MIN-YEAR_MAX, else NaN."""
    if pd.isna(text):
        return np.nan
    for match in YEAR_PATTERN.findall(text):
        year = int(match)
        if YEAR_MIN <= year <= YEAR_MAX:
            return year
    return np.nan


# 1. Clean publication year
df_2023['pub_year'] = df_2023['publicationyear'].map(first_valid_year)

# 2. Coverage after cleaning
usable = df_2023['pub_year'].notna()
print(f"2023 rows: {len(df_2023):,}   2023 checkouts: {total_2023:,}")
print(f"Rows with usable pub_year: {usable.sum():,} ({usable.mean() * 100:.2f}%)")

# 3. Age at checkout, dropping missing years and editions dated after 2023
df_2023['age'] = df_2023['checkoutyear'] - df_2023['pub_year']
negative = df_2023['age'] < 0
aged = df_2023[usable & ~negative]
aged_total = int(aged['checkouts'].sum())
print(f"Rows dropped for negative age (pub_year 2024): {negative.sum():,} "
      f"({int(df_2023.loc[negative, 'checkouts'].sum()):,} checkouts)")
print(f"Checkouts with a usable age: {aged_total:,} ({aged_total / total_2023 * 100:.1f}% of 2023)")

# 4-5. Age buckets
aged = aged.assign(age_bucket=pd.cut(aged['age'], bins=AGE_EDGES, labels=AGE_LABELS))
assert aged['age_bucket'].notna().all()
buckets = aged.groupby('age_bucket', observed=False)['checkouts'].sum().to_frame()
buckets['pct_of_aged'] = buckets['checkouts'] / aged_total * 100
buckets['pct_of_2023_total'] = buckets['checkouts'] / total_2023 * 100
print("\n2023 checkouts by age of edition at checkout:")
print(buckets.to_string(formatters={
    'checkouts': '{:,}'.format, 'pct_of_aged': '{:.1f}%'.format,
    'pct_of_2023_total': '{:.1f}%'.format,
}))

# 6. Headline shares ("10+ years ago" = age 11 or more, matching the bucket edges)
recent = buckets.loc['0-2 yrs']
older = buckets.loc[['11-20 yrs', '21-50 yrs', '50+ yrs']].sum()
print(f"\nLast 2 years (age 0-2):   {recent['pct_of_aged']:.1f}% of aged checkouts "
      f"({recent['pct_of_2023_total']:.1f}% of all 2023)")
print(f"10+ years ago (age 11+):  {older['pct_of_aged']:.1f}% of aged checkouts "
      f"({older['pct_of_2023_total']:.1f}% of all 2023)")

In [ ]:
# Q13: Median checkout age by year — comfort reading test
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# pub_year is cleaned exactly as in Q12 (first standalone 4-digit year in
# 1900-2024); age = checkoutyear - pub_year, with missing and negative ages
# dropped. Every statistic is weighted by `checkouts`, so a row with 40
# checkouts counts 40 times; percentiles are the smallest age whose cumulative
# checkout weight reaches the quantile.
#
# The question: did 2020 readers reach for older books (rereads, backlist,
# comfort reading) more than in 2023?
#
# Caveats carried over from Q12: age is the age of the edition, and digital
# records carry the ebook/audiobook release date, which skews digital young.
# 2020 was 81% digital against 70% in 2023, so the all-formats comparison mixes
# a real reading shift with a format-mix shift. The per-usage-class table below
# separates the two.

import re

import numpy as np
import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
YEAR_MIN, YEAR_MAX = 1900, 2024
YEAR_PATTERN = re.compile(r'(?<!\d)(\d{4})(?!\d)')
QUANTILES = {'p25_age': 0.25, 'median_age': 0.50, 'p75_age': 0.75}
EXPECTED_YEAR_TOTALS = {2020: 3_279_754, 2021: 3_850_336, 2022: 4_361_275, 2023: 4_960_175}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str, 'publicationyear': str}, parse_dates=['date'])
assert df.groupby('checkoutyear')['checkouts'].sum().to_dict() == EXPECTED_YEAR_TOTALS


def first_valid_year(text):
    """First standalone 4-digit number within YEAR_MIN-YEAR_MAX, else NaN (as in Q12)."""
    if pd.isna(text):
        return np.nan
    for match in YEAR_PATTERN.findall(text):
        year = int(match)
        if YEAR_MIN <= year <= YEAR_MAX:
            return year
    return np.nan


df['pub_year'] = df['publicationyear'].map(first_valid_year)
df['age'] = df['checkoutyear'] - df['pub_year']
aged = df[df['age'].notna() & (df['age'] >= 0)]


def weighted_age_stats(group):
    """Checkout-weighted mean and percentiles of age for one group of rows."""
    by_age = group.groupby('age')['checkouts'].sum().sort_index()
    cumulative_share = by_age.cumsum() / by_age.sum()
    stats = {'checkouts': int(by_age.sum()),
             'mean_age': np.average(by_age.index, weights=by_age.to_numpy())}
    for name, q in QUANTILES.items():
        stats[name] = by_age.index[np.searchsorted(cumulative_share.to_numpy(), q)]
    return pd.Series(stats)


# 1. Weighted age profile per checkout year, all formats
coverage = aged.groupby('checkoutyear')['checkouts'].sum() / pd.Series(EXPECTED_YEAR_TOTALS) * 100
by_year = aged.groupby('checkoutyear')[['age', 'checkouts']].apply(weighted_age_stats)
by_year['pct_with_age'] = coverage
columns = ['mean_age', 'p25_age', 'median_age', 'p75_age']

fmt = {'checkouts': '{:,.0f}'.format, 'mean_age': '{:.2f}'.format,
       'p25_age': '{:.0f}'.format, 'median_age': '{:.0f}'.format, 'p75_age': '{:.0f}'.format,
       'pct_with_age': '{:.2f}%'.format}
print("Checkout-weighted age of edition at checkout, all formats:")
print(by_year.to_string(formatters=fmt))

# Year-over-year change
print("\nYear-over-year change (years of age):")
print(by_year[columns].diff().dropna().to_string(float_format='{:+.2f}'.format))

# Same profile within each usage class, to separate reading from format mix
by_class = (
    aged.groupby(['usageclass', 'checkoutyear'])[['age', 'checkouts']]
    .apply(weighted_age_stats)
)
print("\nBy usage class:")
print(by_class.to_string(formatters={k: v for k, v in fmt.items() if k != 'pct_with_age'}))

# Interpretation: 2020 vs 2023
print("\n2020 vs 2023:")
for label, table in [('All formats', by_year),
                     ('Digital', by_class.loc['Digital']),
                     ('Physical', by_class.loc['Physical'])]:
    mean_gap = table.loc[2020, 'mean_age'] - table.loc[2023, 'mean_age']
    median_gap = table.loc[2020, 'median_age'] - table.loc[2023, 'median_age']
    if mean_gap >= 0.5:
        verdict = 'older-skewing in 2020'
    elif mean_gap <= -0.5:
        verdict = 'younger-skewing in 2020'
    else:
        verdict = 'no meaningful difference'
    print(f"  {label:<12} mean {mean_gap:+.2f} yrs, median {median_gap:+.0f} yrs  ->  {verdict}")

In [ ]:
# Q14: Subject surges during COVID vs baseline
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Rows with no subjects are dropped; the comma-separated list is exploded so each
# subject on a row receives that row's full `checkouts`. Subject totals therefore
# add up to more than the overall total: one checkout of a "Fiction, Mystery"
# title counts once for each subject. Duplicate subjects within a row count once.
#
# spike_factor = 2020 checkouts / mean of 2021-2023. Above 1.5 is read as a
# COVID-era surge; sustained growth is 2023 more than 50% above 2020.
#
# Caveat: the two usage classes use different subject vocabularies. Digital rows
# carry coarse OverDrive genres ("Fiction", "Mystery"); Physical rows carry
# granular Library of Congress headings ("Dinosaurs Juvenile literature"). The
# top 50 is dominated by the coarse digital genres, and any subject tied to
# Physical inherits its 2020 closure collapse. pct_digital is printed so this is
# visible, along with the whole-dataset ratio as a benchmark.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 50
SURGE_FACTOR = 1.5
GROWTH_PCT = 50
YEARS = [2020, 2021, 2022, 2023]

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
df = df.dropna(subset=['subjects'])

# 1. One row per (record, subject)
exploded = (
    df[['subjects', 'checkoutyear', 'usageclass', 'checkouts']]
    .assign(subject=df['subjects'].str.split(','))
    .explode('subject')
)
exploded['subject'] = exploded['subject'].str.strip()
exploded = exploded[exploded['subject'] != '']
exploded = exploded.reset_index().drop_duplicates(['index', 'subject'])

# 2. Checkouts per subject and year
by_year = (
    exploded.pivot_table(index='subject', columns='checkoutyear', values='checkouts',
                         aggfunc='sum', fill_value=0)
    .reindex(columns=YEARS, fill_value=0)
)
by_year['total'] = by_year[YEARS].sum(axis=1)
digital = exploded[exploded['usageclass'] == 'Digital'].groupby('subject')['checkouts'].sum()
by_year['pct_digital'] = digital.reindex(by_year.index, fill_value=0) / by_year['total'] * 100

# 3. Spike factor for the top subjects
top = by_year.nlargest(TOP_N, 'total').copy()
top['avg_2021_2023'] = top[[2021, 2022, 2023]].mean(axis=1)
top['spike_factor'] = top[2020] / top['avg_2021_2023']
top['growth_2020_2023_pct'] = (top[2023] / top[2020] - 1) * 100

year_totals = df.groupby('checkoutyear')['checkouts'].sum()
benchmark = year_totals[2020] / year_totals[[2021, 2022, 2023]].mean()
print(f"Subjects: {len(by_year):,}   Top {TOP_N} by 2020-2023 checkouts analysed")
print(f"Benchmark, all rows with subjects: 2020 / avg(2021-2023) = {benchmark:.2f}, "
      f"2023 vs 2020 = {(year_totals[2023] / year_totals[2020] - 1) * 100:+.1f}%")

fmt = {2020: '{:,}'.format, 2021: '{:,}'.format, 2022: '{:,}'.format, 2023: '{:,}'.format,
       'avg_2021_2023': '{:,.0f}'.format, 'spike_factor': '{:.2f}'.format,
       'growth_2020_2023_pct': lambda v: 'new (0 in 2020)' if v == float('inf') else f'{v:+.1f}%',
       'pct_digital': '{:.0f}%'.format}
columns = [2020, 'avg_2021_2023', 'spike_factor', 2023, 'growth_2020_2023_pct', 'pct_digital']

# 4. COVID-era surges and sustained growth
surges = top[top['spike_factor'] > SURGE_FACTOR].sort_values('spike_factor', ascending=False)
print(f"\nCOVID-era surges (spike_factor > {SURGE_FACTOR}): {len(surges)}")
print(surges[columns].to_string(formatters=fmt) if len(surges) else "  none")

growth = top[top['growth_2020_2023_pct'] > GROWTH_PCT].sort_values('growth_2020_2023_pct',
                                                                    ascending=False)
print(f"\nSustained growth (2023 > 2020 by more than {GROWTH_PCT}%): {len(growth)}")
print(growth[columns].to_string(formatters=fmt) if len(growth) else "  none")

print(f"\nAll top {TOP_N}, by spike_factor:")
print(top.sort_values('spike_factor', ascending=False)[columns].to_string(formatters=fmt))

# 5. Top 10 subjects per year
print("\nTop 10 subjects per year:")
ranks = pd.DataFrame({
    year: [f"{s} ({by_year.loc[s, year]:,})" for s in by_year[year].nlargest(10).index]
    for year in YEARS
}, index=pd.RangeIndex(1, 11, name='rank'))
print(ranks.to_string())

In [ ]:
# Q15: Top subject per year
#
# Reads the cleaned file written by Step 5a so this cell stands on its own, and
# explodes subjects exactly as in Q14: rows with no subjects dropped, the comma-
# separated list split and stripped, duplicate subjects within a row counted
# once. Each subject receives its row's full `checkouts`, so a year's subject
# totals add up to more than that year's checkouts.
#
# pct_of_year is the subject's checkouts over the year's total checkouts (all
# rows), i.e. the share of that year's borrowing that carried the subject.
#
# Caveat (see Q14): these coarse genres are OverDrive tags carried only by
# Digital rows; Physical rows use Library of Congress headings, which rarely say
# just "Fiction". The top 5 is therefore effectively a digital ranking, and
# pct_of_year dips in 2021-2022 partly because physical's share of each year rose.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 5
YEARS = [2020, 2021, 2022, 2023]
EXPECTED_YEAR_TOTALS = {2020: 3_279_754, 2021: 3_850_336, 2022: 4_361_275, 2023: 4_960_175}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
year_totals = df.groupby('checkoutyear')['checkouts'].sum()
assert year_totals.to_dict() == EXPECTED_YEAR_TOTALS

with_subjects = df.dropna(subset=['subjects'])
exploded = (
    with_subjects[['subjects', 'checkoutyear', 'checkouts']]
    .assign(subject=with_subjects['subjects'].str.split(','))
    .explode('subject')
)
exploded['subject'] = exploded['subject'].str.strip()
exploded = exploded[exploded['subject'] != '']
exploded = exploded.reset_index().drop_duplicates(['index', 'subject'])

by_year = exploded.groupby(['checkoutyear', 'subject'])['checkouts'].sum()

# Top subjects per year, side by side
table = pd.DataFrame({
    year: [
        f"{subject} ({checkouts:,}, {checkouts / year_totals[year] * 100:.1f}%)"
        for subject, checkouts in by_year.loc[year].nlargest(TOP_N).items()
    ]
    for year in YEARS
}, index=pd.RangeIndex(1, TOP_N + 1, name='rank'))

print(f"Top {TOP_N} subjects per year: checkouts, % of that year's checkouts\n")
print(table.to_string())

In [ ]:
# Q16: Cross-year seasonality — month-of-year patterns
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Each month's checkouts (sum of `checkouts`) are divided by that year's total,
# so a month's share is comparable across years despite 51% growth over the
# period. An even year would put 8.33% in every month.
#
# Spread across the four years is the sample standard deviation (ddof=1) in
# percentage points: under 0.5 is read as a reliable pattern, over 1.5 as
# unreliable, anything between as moderate.
#
# Caveats: 2020 is distorted by the April-July closure (Physical at zero, see
# Q1), so mean_ex_2020 is printed alongside the four-year mean. Months also
# differ in length; February's lower share is partly just 28 days.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
YEARS = [2020, 2021, 2022, 2023]
RELIABLE_STD = 0.5
UNRELIABLE_STD = 1.5
EXPECTED_YEAR_TOTALS = {2020: 3_279_754, 2021: 3_850_336, 2022: 4_361_275, 2023: 4_960_175}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

# 1. Each month's share of its year's total
monthly = df.pivot_table(index='checkoutmonth', columns='checkoutyear', values='checkouts',
                         aggfunc='sum')[YEARS]
assert monthly.shape == (12, 4) and monthly.notna().all().all()
assert monthly.sum().to_dict() == EXPECTED_YEAR_TOTALS
share = monthly / monthly.sum() * 100

# 2-3. Spread of each month's share across the four years
seasonality = pd.DataFrame({
    'mean_share': share.mean(axis=1),
    'std_share': share.std(axis=1, ddof=1),
    'min_share': share.min(axis=1),
    'max_share': share.max(axis=1),
    'mean_ex_2020': share[[2021, 2022, 2023]].mean(axis=1),
})
seasonality['reliability'] = pd.cut(
    seasonality['std_share'], bins=[0, RELIABLE_STD, UNRELIABLE_STD, float('inf')],
    labels=['reliable', 'moderate', 'unreliable'], right=False,
)
seasonality.index = pd.to_datetime(seasonality.index, format='%m').strftime('%b')
seasonality.index.name = 'month'

print("Month's share of its year's checkouts (%), 2020-2023:")
print(seasonality.to_string(float_format='{:.2f}'.format))

print("\nShare by year (%):")
print(share.set_axis(seasonality.index).to_string(float_format='{:.2f}'.format))

# 4. Peak, trough and reliability flags
for label, column in [('four-year mean', 'mean_share'), ('2021-2023 mean', 'mean_ex_2020')]:
    print(f"\nPeak month ({label}):   {seasonality[column].idxmax()} "
          f"({seasonality[column].max():.2f}%)")
    print(f"Trough month ({label}): {seasonality[column].idxmin()} "
          f"({seasonality[column].min():.2f}%)")

for flag, threshold in [('reliable', f'std < {RELIABLE_STD}'),
                        ('unreliable', f'std > {UNRELIABLE_STD}')]:
    months = seasonality.index[seasonality['reliability'] == flag].tolist()
    print(f"\n{flag.capitalize()} ({threshold}): {', '.join(months) if months else 'none'}")

In [ ]:
# Q17: Format seasonality — do physical and digital peak differently?
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Within each usage class, each month's checkouts (sum of `checkouts`) are
# divided by that class's total for the year, then averaged across years. diff
# is physical minus digital, in percentage points.
#
# Caveat: in 2020 Physical had four zero months (April-July closure, see Q1), so
# its 2020 shares are dominated by January-March, and through 2021 it was still
# climbing back (27% to 65% of the Jan 2020 baseline), which inflates late-2021
# months. The 2022-2023 columns use only post-recovery years and are the cleaner
# read on true seasonality; the all-years columns are kept as specified.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
YEARS = [2020, 2021, 2022, 2023]
STEADY_YEARS = [2022, 2023]   # after the physical recovery ramp

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
assert df['checkouts'].sum() == 16_451_540

# 1. Month share of each year's total, within each usage class
monthly = (
    df.pivot_table(index='checkoutmonth', columns=['usageclass', 'checkoutyear'],
                   values='checkouts', aggfunc='sum', fill_value=0)
    .reindex(index=range(1, 13), fill_value=0)
)
share = monthly / monthly.sum() * 100


def mean_share(usage, years):
    """Average of each month's within-year share for one usage class."""
    return share[usage][years].mean(axis=1)


# 2-3. Averaged across years, side by side
table = pd.DataFrame({
    'physical_share': mean_share('Physical', YEARS),
    'digital_share': mean_share('Digital', YEARS),
})
table['diff'] = table['physical_share'] - table['digital_share']
table['physical_2022_23'] = mean_share('Physical', STEADY_YEARS)
table['digital_2022_23'] = mean_share('Digital', STEADY_YEARS)
table['diff_2022_23'] = table['physical_2022_23'] - table['digital_2022_23']
table.index = pd.to_datetime(table.index, format='%m').strftime('%b')
table.index.name = 'month'

print("Mean share of each year's checkouts within usage class (%); diff = physical - digital")
print(table.to_string(float_format='{:.2f}'.format))

# 4. Peak (and trough) months per class
print()
for label, suffix in [('2020-2023', '_share'), ('2022-2023', '_2022_23')]:
    for usage in ['physical', 'digital']:
        column = f'{usage}{suffix}'
        print(f"{label}  {usage.capitalize():<9} peak {table[column].idxmax()} "
              f"({table[column].max():.2f}%), trough {table[column].idxmin()} "
              f"({table[column].min():.2f}%)")

In [ ]:
# Q18: Publisher dominance — digital vs physical
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Rows with no publisher are dropped; % of class total is against the class's
# checkouts that have a publisher. Volume is the sum of `checkouts`.
#
# The two catalogs name publishers differently. Digital (OverDrive) records name
# the publishing group or audio distributor ("Random House, Inc.", "Books on
# Tape"); Physical records name the imprint, with a trailing comma ("Alfred A.
# Knopf,", "Riverhead Books,"). An exact-string overlap of the two top 10s is
# therefore expected to be empty, so overlap is also reported at parent-group
# level using PARENT_GROUPS below: a keyword lookup covering the large groups,
# with anything unmatched kept as its own name. The map is deliberately small
# and visible; it is not a complete imprint registry.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 10

# keyword (lowercase substring) -> parent group; first match wins
PARENT_GROUPS = [
    ('books on tape', 'Penguin Random House'),
    ('random house', 'Penguin Random House'),
    ('penguin', 'Penguin Random House'),
    ('knopf', 'Penguin Random House'),
    ('ballantine', 'Penguin Random House'),
    ('viking', 'Penguin Random House'),
    ('riverhead', 'Penguin Random House'),
    ('doubleday', 'Penguin Random House'),
    ('harper', 'HarperCollins'),
    ('hachette', 'Hachette'),
    ('little, brown', 'Hachette'),
    ('macmillan', 'Macmillan'),
    ('simon & schuster', 'Simon & Schuster'),
    ('simon spotlight', 'Simon & Schuster'),
    ('scholastic', 'Scholastic'),
    ('hyperion', 'Disney'),
]


def parent_group(publisher):
    """Parent group from PARENT_GROUPS, or the cleaned publisher name itself."""
    lowered = publisher.lower()
    for keyword, group in PARENT_GROUPS:
        if keyword in lowered:
            return group
    return publisher.strip().rstrip(',').strip()


df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
df = df.dropna(subset=['publisher'])

# 1. Top 10 publishers per usage class
tops = {}
for usage in ['Digital', 'Physical']:
    by_publisher = df[df['usageclass'] == usage].groupby('publisher')['checkouts'].sum()
    top = by_publisher.nlargest(TOP_N).rename('checkouts').to_frame()
    top['pct_of_class'] = top['checkouts'] / by_publisher.sum() * 100
    top['parent_group'] = [parent_group(p) for p in top.index]
    tops[usage] = top
    print(f"{usage}: top {TOP_N} publishers "
          f"(class checkouts with a publisher: {int(by_publisher.sum()):,}; "
          f"top {TOP_N} hold {top['pct_of_class'].sum():.1f}%)")
    print(top.reset_index().to_string(index=False, formatters={
        'checkouts': '{:,}'.format, 'pct_of_class': '{:.1f}%'.format}))
    print()

# 2-3. Overlap, exact strings and parent groups
exact = sorted(set(tops['Digital'].index) & set(tops['Physical'].index))
print(f"Overlap on exact publisher string: {len(exact)}")
print('  ' + (', '.join(exact) if exact else 'none'))

groups = {usage: set(top['parent_group']) for usage, top in tops.items()}
shared = sorted(groups['Digital'] & groups['Physical'])
print(f"\nOverlap on parent group: {len(shared)}")
for group in shared:
    members = {
        usage: tops[usage].loc[tops[usage]['parent_group'] == group, 'pct_of_class'].sum()
        for usage in tops
    }
    print(f"  {group:<22} Digital top-10 share {members['Digital']:5.1f}%   "
          f"Physical top-10 share {members['Physical']:5.1f}%")
for usage in tops:
    only = sorted(groups[usage] - groups['Physical' if usage == 'Digital' else 'Digital'])
    print(f"  {usage} only: {', '.join(only) if only else 'none'}")

In [52]:
# Q19: Ranking robustness under different edition-collapse rules
#
# Reads the cleaned file written by Step 5a so this cell stands on its own, and
# ranks titles by summed `checkouts` under three keys:
#   1. raw `title`, exactly as catalogued (no normalization)
#   2. clean_title, rebuilt here from `title` with the Step 4b rules and checked
#      against the stored column
#   3. ISBN where available, clean_title otherwise. The `isbn` field is a comma-
#      separated list (ISBN-10 and -13 forms of one edition, sometimes several
#      editions), so the key is its first 13-digit token, else its first token.
#      An ISBN identifies one edition in one format, so this rule collapses less
#      than clean_title, not more.
#
# Important: the source only populates `isbn` from 2022 (0% of 2020-2021 rows,
# about two thirds of 2022, nearly all of 2023; printed below). Rule 3 is
# therefore not a clean edition rule: each work splits into a "[no isbn]" group
# holding its 2020-2021 (and part-2022) checkouts plus one group per ISBN for
# later years, and the "[no isbn]" groups dominate its top 10.
#
# Lists are labelled differently under each rule, so overlap is measured on a
# common identity: every top-10 entry is mapped to a clean_title (raw titles via
# the Step 4b function, ISBN keys via the clean_title carrying most of their
# checkouts). Overlap therefore counts works, not identical strings.
#
# Caveat: Step 4b now merges the print " : " subtitle form, but a print record
# catalogued without a subtitle its digital edition carries still splits (bare
# "braiding sweetgrass"), so rule 2 under-merges a few works.

import re

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
TOP_N = 10

# --- Step 4b clean_title rules, rebuilt verbatim (revised version) ---
SPACED_COLON_RE = re.compile(r'\s+:\s+')
WHITESPACE_RE = re.compile(r'\s+')
ABRIDGED_RE = re.compile(r'\s*\((?:un)?abridged\)')
BRACKETED_RE = re.compile(r'\s*\[[^\]]*\]')
CATALOGUE_SUFFIX_RE = re.compile(r'\s+/\s+.*$')
SERIES_SUFFIX_RE = re.compile(r':\s*(?:[^:]*\bseries,\s*)?book\s+\d+$', re.IGNORECASE)
STOCK_SUBTITLE_RE = re.compile(
    r':\s*(?:a novel|a memoir|a story|a true story|the unabridged edition)$', re.IGNORECASE
)


def clean_title_text(title):
    """Normalise one title string exactly as Step 4b does."""
    if not isinstance(title, str):
        return title
    text = SPACED_COLON_RE.sub(': ', title)
    text = WHITESPACE_RE.sub(' ', text)
    text = text.lower()
    text = ABRIDGED_RE.sub('', text)
    text = BRACKETED_RE.sub('', text)
    text = CATALOGUE_SUFFIX_RE.sub('', text)
    text = text.strip().rstrip('.').strip()
    text = SERIES_SUFFIX_RE.sub('', text)
    text = STOCK_SUBTITLE_RE.sub('', text)
    text = text.strip().rstrip(':.').strip()
    return text or title.lower().strip()


def isbn_key(isbn_list):
    """First ISBN-13 in a comma-separated list, else the first entry."""
    tokens = [t.strip() for t in isbn_list.split(',') if t.strip()]
    return next((t for t in tokens if len(t) == 13 and t.isdigit()), tokens[0])


df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])
assert df['checkouts'].sum() == 16_451_540

distinct_titles = df['title'].dropna().unique()
rebuilt = df['title'].map(pd.Series([clean_title_text(t) for t in distinct_titles],
                                    index=distinct_titles))
mismatches = (rebuilt != df['clean_title']).sum()
assert mismatches == 0, f"{mismatches} rows differ from the stored clean_title"
print("Rebuilt clean_title matches the stored column on every row.")

has_isbn = df['isbn'].notna()
df['isbn_rule_key'] = df['clean_title'].radd('title:')
df.loc[has_isbn, 'isbn_rule_key'] = 'isbn:' + df.loc[has_isbn, 'isbn'].map(isbn_key)
print(f"Rows with an ISBN: {has_isbn.mean() * 100:.1f}% "
      f"({df.loc[has_isbn, 'checkouts'].sum() / df['checkouts'].sum() * 100:.1f}% of checkouts)")
coverage = df.assign(has_isbn=has_isbn).pivot_table(
    index='checkoutyear', columns='materialtype', values='has_isbn', aggfunc='mean') * 100
print("Share of rows with an ISBN, by checkout year (%):")
print(coverage.to_string(float_format='{:.0f}'.format) + "\n")


def top_by(key):
    """Top-N keys by summed checkouts."""
    return df.groupby(key)['checkouts'].sum().nlargest(TOP_N)


def dominant(frame_key, value):
    """Per key, the value of `value` carrying the most checkouts."""
    return (df.groupby([frame_key, value])['checkouts'].sum()
              .sort_values(ascending=False).reset_index()
              .drop_duplicates(frame_key).set_index(frame_key)[value])


top_raw = top_by('title')
top_clean = top_by('clean_title')
top_isbn = top_by('isbn_rule_key')

# Human-readable labels and common identity for the ISBN rule
isbn_title = dominant('isbn_rule_key', 'clean_title')
isbn_format = dominant('isbn_rule_key', 'materialtype')
isbn_labels = [
    f"{isbn_title[k]} [{isbn_format[k].lower()}, {k.split(':', 1)[1]}]" if k.startswith('isbn:')
    else f"{isbn_title[k]} [no isbn]"
    for k in top_isbn.index
]

side_by_side = pd.DataFrame({
    '1. raw title': [f"{t} ({c:,})" for t, c in top_raw.items()],
    '2. clean_title': [f"{t} ({c:,})" for t, c in top_clean.items()],
    '3. isbn else clean_title': [f"{l} ({c:,})" for l, c in zip(isbn_labels, top_isbn)],
}, index=pd.RangeIndex(1, TOP_N + 1, name='rank'))
print(side_by_side.to_string())

# Overlap on the common clean_title identity
works = {
    'raw': {clean_title_text(t) for t in top_raw.index},
    'clean': set(top_clean.index),
    'isbn': {isbn_title[k] for k in top_isbn.index},
}
in_all = works['raw'] & works['clean'] & works['isbn']
print(f"\nDistinct works in each top {TOP_N}: raw {len(works['raw'])}, "
      f"clean {len(works['clean'])}, isbn {len(works['isbn'])}")
print(f"Works in all three top-{TOP_N} lists: {len(in_all)}")
print('  ' + (', '.join(sorted(in_all)) if in_all else 'none'))
for a, b in [('raw', 'clean'), ('clean', 'isbn'), ('raw', 'isbn')]:
    print(f"Overlap {a} vs {b}: {len(works[a] & works[b])}")

Rebuilt clean_title matches the stored column on every row.
Rows with an ISBN: 47.9% (48.0% of checkouts)
Share of rows with an ISBN, by checkout year (%):
materialtype  AUDIOBOOK  BOOK  EBOOK
checkoutyear                        
2020                  0     0      0
2021                  0     0      0
2022                 70    66     67
2023                 99   100    100

                                                                                                        1. raw title                                                                                     2. clean_title                                                                                     3. isbn else clean_title
rank                                                                                                                                                                                                                                                                                                    

In [ ]:
# Q20: Dataset coverage against SPL public reporting
#
# Reads the cleaned file written by Step 5a so this cell stands on its own.
# Annual totals are the sum of `checkouts` per checkout year.
#
# SPL_ANNUAL holds the library's publicly reported annual circulation, to be
# filled in from the spl.org annual reports. Coverage is printed only for years
# that have a value.
#
# Expect coverage well under 100% by construction: this extract keeps only
# BOOK, EBOOK and AUDIOBOOK and only title-months with five or more checkouts,
# while reported circulation typically counts every material type (DVDs, music,
# magazines, devices) and may include renewals. Check how each report defines
# circulation before comparing.

import pandas as pd
from pathlib import Path

CLEAN_PATH = Path('../Data/Clean/seattle_checkouts_2020_2023_clean.csv')
EXPECTED_YEAR_TOTALS = {2020: 3_279_754, 2021: 3_850_336, 2022: 4_361_275, 2023: 4_960_175}

# Note: these are placeholder values - will verify with web search
SPL_ANNUAL = {2020: None, 2021: None, 2022: None, 2023: None}

df = pd.read_csv(CLEAN_PATH, dtype={'isbn': str}, parse_dates=['date'])

# 1. Dataset totals per year
annual = df.pivot_table(index='checkoutyear', columns='usageclass', values='checkouts',
                        aggfunc='sum')[['Digital', 'Physical']]
annual['dataset_total'] = annual.sum(axis=1)
assert annual['dataset_total'].to_dict() == EXPECTED_YEAR_TOTALS

# 2-3. Coverage against SPL-reported circulation, where available
annual['spl_reported'] = pd.Series(SPL_ANNUAL, dtype='Int64')
annual['coverage_pct'] = annual['dataset_total'] / annual['spl_reported'] * 100

print("Dataset checkouts per year (BOOK/EBOOK/AUDIOBOOK, title-months with >= 5 checkouts):")
display = pd.DataFrame({
    column: annual[column].map('{:,}'.format)
    for column in ['Digital', 'Physical', 'dataset_total']
})
display['spl_reported'] = annual['spl_reported'].map(
    lambda v: 'not yet filled' if pd.isna(v) else f'{int(v):,}')
display['coverage_pct'] = annual['coverage_pct'].map(
    lambda v: '-' if pd.isna(v) else f'{v:.1f}%')
print(display.to_string())
print(f"\nDataset total 2020-2023: {int(annual['dataset_total'].sum()):,}")

missing = [year for year, value in SPL_ANNUAL.items() if value is None]
if missing:
    print(f"SPL-reported circulation still to fill in for: {', '.join(map(str, missing))}")